# EEG Channel Estimation — Non-Wearable from Wearable
Estimating non-wearable EEG channel amplitude (top-5 stress-correlated) \
from 4 wearable channels (Fp1, Fp2, F7, F8) using 2 input modes:

| Mode | Input Features | N Features |
|------|---------------|------------|
| `original`   | Actual signal only                     | 4  |
| `derivative` | Actual signal + first-order derivative | 8  |

**Models:** ANN (MLP), Random Forest, BDTR, SVR  
**Evaluation:** Pearson R, RMSE, MAE, NSE  
**Cross-validation:** Leave-One-Trial-Out (LOTO) — 3 rotations, result = average

---
**Notebook structure:**
1. Imports & Setup
2. Configuration
3. Utility Functions
4. Feature Engineering
5. Visualization Functions
6. Save / Load Training Results
7. Signal Export
8. LOTO Training
9. Multi-Subject Summary
10. Execution
11. Mode Comparison
12. Generate Visualizations from Checkpoint
13. EEG Electrode Position Map
14. Spatial Distance Analysis

## 1. Imports & Setup

In [ ]:
# -*- coding: utf-8 -*-
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import scipy.io as sio
from scipy.stats import pearsonr
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
warnings.filterwarnings('ignore')

print("GPU available:", tf.config.list_physical_devices('GPU'))

from google.colab import drive
drive.mount('/content/drive')

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configuration

In [ ]:
# =============================================================================
# Channel & model configuration
# =============================================================================
CHANNEL_NAMES = [
    'CZ','FZ','Fp1','F7','F3','FC1','C3','FC5',
    'FT9','T7','CP5','CP1','P3','P7','PO9','O1',
    'PZ','OZ','O2','PO10','P8','P4','CP2','CP6',
    'T8','FT10','FC6','C4','FC2','F4','F8','Fp2'
]

WEARABLE_CH = ['Fp1', 'Fp2', 'F7', 'F8']

# Top-5 non-wearable target channels (selected by stress-feature correlation)
NEW_ESTIMATION_SCHEME = {
    'FC2' : WEARABLE_CH,   # rank 2 — max_abs_r = 0.2221
    'FZ'  : WEARABLE_CH,   # rank 3 — max_abs_r = 0.2172
    'CP2' : WEARABLE_CH,   # rank 4 — max_abs_r = 0.2112
    'CZ'  : WEARABLE_CH,   # rank 5 — max_abs_r = 0.2105
    'P4'  : WEARABLE_CH,   # rank 7 — max_abs_r = 0.2099
}

MODEL_COLORS = {
    'ANN_MLP'      : '#E63946',
    'Random_Forest': '#2A9D8F',
    'BDTR'         : '#F4A261',
    'SVR'          : '#9B5DE5',
}

SAMPLING_RATE = 128
TASKS         = ['Arithmetic', 'Stroop', 'Mirror_image']
N_TRIALS      = 3

# =============================================================================
# Paths — Google Drive
# =============================================================================
DATA_DIR   = '/content/drive/MyDrive/Thesis/dataset/SAM40/filteredData'
OUTPUT_DIR = '/content/drive/MyDrive/Thesis/channel_estimation'
PAPER_DIR  = os.path.join(OUTPUT_DIR, 'paper_figures')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# Visualization constants
# =============================================================================
VIZ_SPLIT = 'split_2'   # last LOTO split (trial 3 as test) for timeseries & scatter

VIZ_DIRS = {
    'timeseries' : 'timeseries',
    'scatter'    : 'scatter',
    'r_bar'      : 'r_bar',
    'overfitting': 'overfitting',
    'distance'   : 'distance',
}

MODE_DISPLAY = {
    'original'  : 'Actual Signal',
    'derivative': 'Actual + Derivative Signal'
}
MODE_COLOR = {'original': '#185FA5', 'derivative': '#D85A30'}
MODE_LS    = {'original': '-',       'derivative': '--'}

# Pre-computed mean Euclidean distance from each target channel
# to the 4 wearable channels in polar coordinate units
MEAN_DIST = {
    'FC2': 0.4501, 'FZ': 0.3617, 'CP2': 0.6564,
    'CZ' : 0.5360, 'P4': 0.7946
}

PAPER_DPI = 1000

## 3. Utility Functions

In [ ]:
import scipy.io as sio

MAT_KEY = 'Clean_data'

def load_mat(filepath):
    mat  = sio.loadmat(filepath)
    data = mat[MAT_KEY].astype(np.float64)  # shape: (32, 3200)
    df   = pd.DataFrame(data.T, columns=CHANNEL_NAMES)  # -> (3200, 32)
    return df


def normalize_data(train_data, test_data):
    scaler     = MinMaxScaler(feature_range=(-1, 1))
    train_norm = scaler.fit_transform(train_data)
    test_norm  = scaler.transform(test_data)
    return train_norm, test_norm, scaler


def compute_metrics(y_true, y_pred):
    r, _  = pearsonr(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mae   = mean_absolute_error(y_true, y_pred)
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((y_true - np.mean(y_true)) ** 2)
    nse   = 1 - (numer / denom) if denom != 0 else np.nan
    return {'R': round(r, 4), 'RMSE': round(rmse, 6),
            'MAE': round(mae, 6), 'NSE': round(nse, 4)}


def build_models():
    return {
        'ANN_MLP': MLPRegressor(
            hidden_layer_sizes=(100, 100, 100),
            activation='tanh', solver='adam',
            max_iter=500, random_state=42,
            early_stopping=True, validation_fraction=0.1
        ),
        'Random_Forest': RandomForestRegressor(
            n_estimators=100, random_state=42, n_jobs=-1
        ),
        'BDTR': GradientBoostingRegressor(
            n_estimators=100, learning_rate=0.15,
            max_depth=3, random_state=42
        ),
        'SVR': SVR(kernel='rbf', C=10, epsilon=0.1, gamma=0.5),
    }


def train_keras_ann(X_train, y_train, X_test, y_test_orig, scaler_y, input_dim):
    """
    Train ANN with multiple architecture and learning rate combinations
    (trial-and-error), following Essam et al. (2021) methodology.
    Select the model with the highest test R.
    """
    architectures  = [(256, 128, 64), (256, 256, 128, 64), (512, 256, 128)]
    learning_rates = [0.001, 0.0005, 0.0001]

    best_model  = None
    best_r      = -np.inf
    best_params = {}
    lr_history  = {}

    start_ann = time.time()        # ← TAMBAHKAN INI
    combo_count = 0                # ← TAMBAHKAN INI

    for lr in learning_rates:
        for arch in architectures:
            combo_count += 1       # ← TAMBAHKAN INI
            start_combo = time.time()  # ← TAMBAHKAN INI
            model = keras.Sequential()
            model.add(layers.Input(shape=(input_dim,)))
            for units in arch:
                model.add(layers.Dense(units, activation='tanh'))
            model.add(layers.Dense(1))
            model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                          loss='mse')
            model.fit(
                X_train, y_train,
                epochs=200, batch_size=64, validation_split=0.1,
                callbacks=[keras.callbacks.EarlyStopping(
                    patience=15, restore_best_weights=True)],
                verbose=0
            )
            y_pred = scaler_y.inverse_transform(
                model.predict(X_test, verbose=0)
            ).ravel()
            r, _ = pearsonr(y_test_orig, y_pred)

            if lr not in lr_history or r > lr_history[lr]:
                lr_history[lr] = round(float(r), 4)

            if r > best_r:
                best_r      = r
                best_model  = model
                best_params = {
                    'architecture'   : arch,
                    'n_layers'       : len(arch),
                    'nodes_per_layer': arch[0],
                    'learning_rate'  : lr,
                    'activation'     : 'tanh'
                }
    print(f"    [TIMING] ANN grid search total: {time.time()-start_ann:.1f}s "
          f"| Best R={best_r:.4f} | Best arch={best_params['architecture']} "
          f"lr={best_params['learning_rate']}")

    return best_model, best_params, lr_history

## 4. Feature Engineering

In [ ]:
def expand_features(df, channels, mode='original'):
    """
    Build feature matrix from wearable signals based on mode:

        'original'   -> actual signal only                    (4 features)
                        [Fp1, Fp2, F7, F8]

        'derivative' -> actual signal + first-order derivative (8 features)
                        [Fp1, Fp2, F7, F8,
                         diff1_Fp1, diff1_Fp2, diff1_F7, diff1_F8]
                        Derivative computed using central difference:
                        diff1_V[t] = (V[t+1] - V[t-1]) / (2 * dt)

    Parameters:
        df       : DataFrame with channel columns
        channels : list of wearable channel names
        mode     : 'original' | 'derivative'

    Returns:
        df_out    : feature DataFrame
        col_names : list of column names
    """
    VALID_MODES = ('original', 'derivative')
    if mode not in VALID_MODES:
        raise ValueError(f"mode must be one of {VALID_MODES}, got: '{mode}'")

    dt     = 1 / SAMPLING_RATE
    df_out = df[list(channels)].copy()

    if mode == 'derivative':
        for ch in channels:
            df_out[f'diff1_{ch}'] = np.gradient(df[ch].values, dt)

    col_names = list(df_out.columns)
    return df_out, col_names

## 5. Visualization Functions

All visualization functions are defined here. Call `generate_all()` after training
to produce all figures, or `generate_paper_figures()` for publication-ready figures.

**Full output per subject (69 figures):**
```
plots_sub{id}/
├── timeseries/     → 30 (5ch × 3task × 2mode)
├── scatter/        → 30 (5ch × 3task × 2mode, 4 model subplots each)
├── r_bar/          →  6 (3task × 2mode)
├── overfitting/    →  2 (1 per mode)
└── distance/       →  1 (both modes)
```

**Paper figures per subject (~10 figures):** auto-selected based on best model/channel/task.

In [ ]:
import matplotlib.patheffects as _pe

# =============================================================================
# Helpers
# =============================================================================

def _load_avg_csv(subject_id, input_mode):
    """Load averaged results CSV for a subject."""
    label = {'original': 'Actual', 'derivative': 'Derivative'}[input_mode]
    path  = os.path.join(OUTPUT_DIR,
                         f"results_{label}_sub{subject_id}_averaged.csv")
    if not os.path.exists(path):
        print(f"  [WARN] Not found: {path}")
        return None
    return pd.read_csv(path)


def _savefig(fig, fpath, dpi=1000):
    os.makedirs(os.path.dirname(fpath), exist_ok=True)
    fig.savefig(fpath, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  Saved: {os.path.relpath(fpath, OUTPUT_DIR)}")


# =============================================================================
# 1. Timeseries — 5ch x 3task x 2mode = 30 figures per subject
# =============================================================================

def plot_timeseries(cache_data, subject_id, input_mode,
                    split_key=VIZ_SPLIT, n_display=640, save_dir=None):
    """
    Plot actual vs estimated timeseries for every channel x task combination.
    Actual signal linewidth = 1.0 so estimated lines remain readable.
    Produces 5ch x 3task = 15 figures per mode call (30 total per subject).
    """
    if save_dir is None:
        save_dir = os.path.join(OUTPUT_DIR, f'plots_sub{subject_id}')
    label = {'original': 'Actual', 'derivative': 'Derivative'}[input_mode]
    saved = []

    for task in TASKS:
        for target_ch in NEW_ESTIMATION_SCHEME:
            ch_cache = (cache_data.get(task, {})
                                  .get(split_key, {})
                                  .get(target_ch))
            if ch_cache is None:
                continue

            y_true      = ch_cache['y_test_orig']
            predictions = ch_cache['predictions']
            n_pts       = min(n_display, len(y_true))
            time_axis   = np.arange(n_pts) / SAMPLING_RATE
            duration    = time_axis[-1]

            fig, ax = plt.subplots(figsize=(16, 4))
            fig.patch.set_facecolor('#FAFAFA')

            for sec in range(0, int(duration) + 1, 2):
                ax.axvspan(sec, min(sec + 1, duration),
                           color='#E8EAF6', alpha=0.4, zorder=1)
            for sec in range(1, int(duration) + 1):
                ax.axvline(sec, color='#BBBBCC', linewidth=0.4,
                           linestyle='-', alpha=0.5, zorder=2)

            ax.plot(time_axis, y_true[:n_pts], color='#1A1A2E',
                    linewidth=1.0, alpha=0.88, label='Actual', zorder=5)
            for model_name, y_pred in predictions.items():
                ax.plot(time_axis, y_pred[:n_pts],
                        color=MODEL_COLORS.get(model_name, '#888'),
                        linewidth=0.85, alpha=0.78,
                        label=model_name, zorder=4)

            ax.set_xlim(0, duration)
            ax.set_xticks(np.arange(0, duration + 1, 2))
            ax.set_xlabel('Time (s)', fontsize=11)
            ax.set_ylabel('Amplitude (\u00b5V)', fontsize=11)
            ax.set_title(
                f'Actual vs Estimated \u2014 Channel {target_ch} | '
                f'Task: {task.replace("_", " ")}\n'
                f'Input: {MODE_DISPLAY[input_mode]} | Subject: {subject_id}',
                fontsize=11, fontweight='bold'
            )
            ax.set_facecolor('#F4F6FF')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.legend(handles=[
                Line2D([0],[0], color='#1A1A2E', linewidth=1.0, label='Actual')
            ] + [
                Line2D([0],[0], color=MODEL_COLORS[m], linewidth=0.85, label=m)
                for m in predictions
            ], loc='upper right', fontsize=9, framealpha=0.88,
               ncol=len(predictions)+1)

            plt.tight_layout()
            ts_dir = os.path.join(save_dir, VIZ_DIRS['timeseries'])
            _savefig(fig, os.path.join(ts_dir, f"{label}_{task}_{target_ch}.png"))
            saved.append(os.path.join(ts_dir, f"{label}_{task}_{target_ch}.png"))

    return saved


# =============================================================================
# 2. Scatter — 5ch x 3task x 2mode = 30 figures, 4 model subplots each
# =============================================================================

def plot_scatter(cache_data, subject_id, input_mode,
                 split_key=VIZ_SPLIT, save_dir=None):
    """
    Produce 15 figures per mode (5ch x 3task), each with 4 scatter subplots.
    Total: 30 figures per subject.
    """
    if save_dir is None:
        save_dir = os.path.join(OUTPUT_DIR, f'plots_sub{subject_id}')
    label      = {'original': 'Actual', 'derivative': 'Derivative'}[input_mode]
    model_list = list(MODEL_COLORS.keys())
    saved      = []

    for task in TASKS:
        task_cache = cache_data.get(task, {}).get(split_key, {})
        for target_ch in NEW_ESTIMATION_SCHEME:
            ch_cache = task_cache.get(target_ch)
            if ch_cache is None:
                continue

            y_true      = ch_cache['y_test_orig']
            predictions = ch_cache['predictions']

            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            fig.patch.set_facecolor('#FAFAFA')
            fig.suptitle(
                f'Scatter \u2014 Channel: {target_ch} | Task: {task.replace("_"," ")}\n'
                f'Input: {MODE_DISPLAY[input_mode]} | Subject: {subject_id}',
                fontsize=11, fontweight='bold'
            )

            for ax, model_name in zip(axes, model_list):
                color  = MODEL_COLORS.get(model_name, '#888')
                y_pred = predictions.get(model_name)
                if y_pred is None:
                    ax.set_visible(False)
                    continue
                metrics = compute_metrics(y_true, y_pred)
                ax.scatter(y_true, y_pred, color=color,
                           alpha=0.25, s=4, edgecolors='none', zorder=3)
                vmin = min(y_true.min(), y_pred.min())
                vmax = max(y_true.max(), y_pred.max())
                ax.plot([vmin, vmax], [vmin, vmax], color='#1A1A2E',
                        linewidth=1.5, linestyle='--', alpha=0.8, zorder=4)
                ax.text(0.04, 0.97,
                        f"R    = {metrics['R']:.4f}\n"
                        f"RMSE = {metrics['RMSE']:.4f}\n"
                        f"MAE  = {metrics['MAE']:.4f}\n"
                        f"NSE  = {metrics['NSE']:.4f}",
                        transform=ax.transAxes, fontsize=8,
                        verticalalignment='top', fontfamily='monospace',
                        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                                  alpha=0.88, edgecolor=color, linewidth=1.2))
                ax.set_xlabel('Actual (\u00b5V)', fontsize=10)
                ax.set_ylabel('Estimated (\u00b5V)', fontsize=10)
                ax.set_title(model_name, fontsize=10,
                             fontweight='bold', color=color)
                ax.set_facecolor('#F8F8F8')
                ax.grid(True, linestyle='--', alpha=0.35, color='#CCCCCC')
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                ax.set_aspect('equal', adjustable='box')

            plt.tight_layout()
            sc_dir = os.path.join(save_dir, VIZ_DIRS['scatter'])
            fpath  = os.path.join(sc_dir, f"{label}_{task}_{target_ch}.png")
            _savefig(fig, fpath)
            saved.append(fpath)

    return saved


# =============================================================================
# 3. R Bar — 3task x 2mode = 6 figures
# =============================================================================

def plot_r_bar(cache_data, subject_id, input_mode,
               split_key=VIZ_SPLIT, save_dir=None):
    """Produce 3 R bar charts (one per task) for a given mode."""
    if save_dir is None:
        save_dir = os.path.join(OUTPUT_DIR, f'plots_sub{subject_id}')
    label      = {'original': 'Actual', 'derivative': 'Derivative'}[input_mode]
    channels   = list(NEW_ESTIMATION_SCHEME.keys())
    model_list = list(MODEL_COLORS.keys())
    n_ch, n_m  = len(channels), len(model_list)
    bar_width  = 0.18
    x          = np.arange(n_ch)
    saved      = []

    for task in TASKS:
        task_cache = cache_data.get(task, {}).get(split_key, {})
        fig, ax = plt.subplots(figsize=(9, 5))
        fig.patch.set_facecolor('#FAFAFA')

        for i, model_name in enumerate(model_list):
            r_values = [
                task_cache.get(ch, {}).get('metrics_test', {})
                          .get(model_name, {}).get('R', 0)
                for ch in channels
            ]
            offset = (i - n_m / 2 + 0.5) * bar_width
            bars   = ax.bar(x + offset, r_values, bar_width,
                            label=model_name, color=MODEL_COLORS[model_name],
                            alpha=0.85, edgecolor='white', linewidth=0.5)
            for bar, r in zip(bars, r_values):
                if r > 0.05:
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_height() + 0.01, f'{r:.2f}',
                            ha='center', va='bottom', fontsize=6.5, rotation=90)

        ax.axhline(0.7, color='#AAAAAA', linewidth=1.2,
                   linestyle='--', alpha=0.7, label='R = 0.7')
        ax.set_xticks(x)
        ax.set_xticklabels(channels, fontsize=10)
        ax.set_ylim(0, 1.18)
        ax.set_ylabel('R', fontsize=11)
        ax.set_xlabel('Target Channel', fontsize=11)
        ax.set_title(
            f'R per Model per Channel | Task: {task.replace("_"," ")}\n'
            f'Input: {MODE_DISPLAY[input_mode]} | Subject: {subject_id}',
            fontsize=11, fontweight='bold'
        )
        ax.set_facecolor('#F8F8F8')
        ax.grid(True, axis='y', linestyle='--', alpha=0.35)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.legend(fontsize=8.5, loc='upper right', framealpha=0.88)
        plt.tight_layout()

        rb_dir = os.path.join(save_dir, VIZ_DIRS['r_bar'])
        fpath  = os.path.join(rb_dir, f"{label}_{task}.png")
        _savefig(fig, fpath)
        saved.append(fpath)

    return saved


# =============================================================================
# 4. Overfitting Detection — 1 per mode = 2 per subject
# =============================================================================

def plot_overfitting(cache_data, subject_id, input_mode,
                     split_key=VIZ_SPLIT, save_dir=None):
    """Scatter R_train vs R_test and NSE_train vs NSE_test."""
    if save_dir is None:
        save_dir = os.path.join(OUTPUT_DIR, f'plots_sub{subject_id}')
    label        = {'original': 'Actual', 'derivative': 'Derivative'}[input_mode]
    TASK_MARKERS = {'Arithmetic': 'o', 'Stroop': 's', 'Mirror_image': '^'}
    model_list   = list(MODEL_COLORS.keys())

    records = []
    for task in TASKS:
        for ch, ch_data in cache_data.get(task, {}).get(split_key, {}).items():
            m_train = ch_data.get('metrics_train', {})
            m_test  = ch_data.get('metrics_test',  {})
            for mn in model_list:
                if mn not in m_train or mn not in m_test:
                    continue
                records.append({
                    'Task': task, 'Channel': ch, 'Model': mn,
                    'R_train': m_train[mn]['R'],   'R_test': m_test[mn]['R'],
                    'NSE_train': m_train[mn]['NSE'], 'NSE_test': m_test[mn]['NSE'],
                })

    if not records:
        print("  [SKIP] No training metrics for overfitting plot.")
        return

    df = pd.DataFrame(records)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor('#FAFAFA')
    fig.suptitle(
        f'Overfitting Detection | Subject: {subject_id} | '        f'Input: {MODE_DISPLAY[input_mode]}',
        fontsize=12, fontweight='bold'
    )

    for ax, metric in zip(axes, ['R', 'NSE']):
        ct, ce = f'{metric}_train', f'{metric}_test'
        all_v  = pd.concat([df[ct], df[ce]])
        vmin, vmax = all_v.min() - 0.05, all_v.max() + 0.05
        ax.plot([vmin, vmax], [vmin, vmax], color='#1A1A2E',
                linewidth=1.5, linestyle='--', alpha=0.7, zorder=1)
        ax.fill_between([vmin, vmax], [vmin-.1, vmax-.1], [vmin+.1, vmax+.1],
                        color='#C8E6C9', alpha=0.3, zorder=0)
        for mn in model_list:
            for task, marker in TASK_MARKERS.items():
                sub = df[(df['Model']==mn) & (df['Task']==task)]
                if sub.empty: continue
                ax.scatter(sub[ct], sub[ce], color=MODEL_COLORS[mn],
                           marker=marker, s=70, alpha=0.85,
                           edgecolors='white', linewidths=0.5, zorder=3)
        ax.set_xlim(vmin, vmax); ax.set_ylim(vmin, vmax)
        ax.set_xlabel(f'{metric} Training', fontsize=10)
        ax.set_ylabel(f'{metric} Test', fontsize=10)
        ax.set_title(f'{metric}: Train vs Test', fontsize=10, fontweight='bold')
        ax.set_facecolor('#F8F8F8')
        ax.grid(True, linestyle='--', alpha=0.35)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.set_aspect('equal', adjustable='datalim')
        leg_model = [Line2D([0],[0], marker='o', color='w',
                             markerfacecolor=MODEL_COLORS[m],
                             markersize=8, label=m) for m in model_list]
        leg_task  = [Line2D([0],[0], marker=mk, color='#555',
                             markersize=8, linestyle='None', label=t)
                     for t, mk in TASK_MARKERS.items()]
        leg_extra = [
            Line2D([0],[0], color='#1A1A2E', linewidth=1.5,
                   linestyle='--', label='Ideal (train=test)'),
            plt.Rectangle((0,0),1,1, fc='#C8E6C9', alpha=0.5, label='Gap \u00b10.1')
        ]
        ax.legend(handles=leg_model+leg_task+leg_extra,
                  fontsize=7.5, loc='lower right', framealpha=0.88, ncol=2)

    plt.tight_layout()
    ov_dir = os.path.join(save_dir, VIZ_DIRS['overfitting'])
    fpath  = os.path.join(ov_dir, f"{label}.png")
    _savefig(fig, fpath)
    return fpath


# =============================================================================
# 5. Distance vs R — 1 figure per subject (both modes combined)
# =============================================================================

def plot_distance_vs_r(subject_id, df_orig, df_deriv, save_dir=None):
    """Mean spatial distance vs mean Pearson R, both modes on one figure."""
    if save_dir is None:
        save_dir = os.path.join(OUTPUT_DIR, f'plots_sub{subject_id}')
    channels  = list(MEAN_DIST.keys())
    distances = [MEAN_DIST[ch] for ch in channels]
    offsets   = {
        'FZ' : (-0.02, 0.022), 'FC2': (0.01, 0.018),
        'CZ' : (0.01, -0.030), 'CP2': (-0.02, 0.022),
        'P4' : (0.01, 0.018)
    }
    cfg = {
        'original'  : {'df': df_orig,  'color': '#185FA5', 'marker': 'o',
                       'label': 'Actual Signal'},
        'derivative': {'df': df_deriv, 'color': '#D85A30', 'marker': 's',
                       'label': 'Actual + Derivative Signal'},
    }
    fig, ax = plt.subplots(figsize=(8, 6))
    fig.patch.set_facecolor('#FAFAFA')
    annotated = False

    for mode, c in cfg.items():
        df = c['df']
        if df is None or df.empty:
            continue
        mean_r = df.groupby('Target_CH')['R'].mean().reindex(channels)
        valid  = ~mean_r.isna()
        if valid.sum() >= 2:
            d_v = np.array(distances)[valid.values]
            r_v = mean_r.fillna(0).values[valid.values]
            x_l = np.linspace(min(distances)-.03, max(distances)+.03, 100)
            ax.plot(x_l, np.poly1d(np.polyfit(d_v, r_v, 1))(x_l),
                    color=c['color'], linewidth=1.2, linestyle='--', alpha=0.65)
        ax.scatter(distances, mean_r.fillna(0).values,
                   color=c['color'], marker=c['marker'],
                   s=110, zorder=5, edgecolors='white',
                   linewidths=0.8, label=c['label'])
        if not annotated and valid.any():
            for i, ch in enumerate(channels):
                ox, oy = offsets.get(ch, (0.01, 0.018))
                ax.annotate(ch,
                    xy=(distances[i], mean_r.fillna(0).values[i]),
                    xytext=(distances[i]+ox, mean_r.fillna(0).values[i]+oy),
                    fontsize=9, fontweight='bold', color='#333',
                    ha='center', va='bottom',
                    path_effects=[_pe.withStroke(linewidth=2, foreground='white')])
            annotated = True

    ax.axhline(0.7, color='#888', linewidth=1.0, linestyle=':',
               zorder=2, label='R = 0.70 threshold')
    ax.set_xlabel('Mean Distance to Wearable Channels (polar units)', fontsize=11)
    ax.set_ylabel('Mean R (all models & tasks)', fontsize=11)
    ax.set_title(
        f'Spatial Distance vs. Estimation Accuracy\nSubject: {subject_id} | LOTO Averaged',
        fontsize=11, fontweight='bold'
    )
    ax.grid(True, linestyle='--', alpha=0.35, color='#CCCCCC')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.legend(fontsize=9, loc='upper right', framealpha=0.9)
    ax.set_facecolor('#F8F8F8')
    plt.tight_layout()
    dist_dir = os.path.join(save_dir, VIZ_DIRS['distance'])
    fpath    = os.path.join(dist_dir, 'distance_vs_R.png')
    _savefig(fig, fpath)
    return fpath


# =============================================================================
# generate_all — run all 5 visualization types for a list of subjects
# =============================================================================

def generate_all(subject_ids=None):
    """
    Generate all 69 figures per subject from training cache.
    Call after training is complete.
    """
    if subject_ids is None:
        subject_ids = SUBJECT_IDS

    for sid in subject_ids:
        print(f"\n{'#'*60}")
        print(f"# Generating visualizations | Subject: {sid}")
        print(f"{'#'*60}")
        save_dir = os.path.join(OUTPUT_DIR, f'plots_sub{sid}')

        for mode in ['original', 'derivative']:
            cache = load_training_results(sid, mode)
            if cache is None:
                print(f"  [SKIP] Cache not found for sub{sid} mode={mode}")
                continue
            print(f"\n  Mode: {mode}")
            print(f"  [1/4] Timeseries...")
            plot_timeseries(cache, sid, mode, save_dir=save_dir)
            print(f"  [2/4] Scatter...")
            plot_scatter(cache, sid, mode, save_dir=save_dir)
            print(f"  [3/4] R bar...")
            plot_r_bar(cache, sid, mode, save_dir=save_dir)
            print(f"  [4/4] Overfitting detection...")
            plot_overfitting(cache, sid, mode, save_dir=save_dir)

        print(f"  [5/5] Distance vs R...")
        df_o = _load_avg_csv(sid, 'original')
        df_d = _load_avg_csv(sid, 'derivative')
        plot_distance_vs_r(sid, df_o, df_d, save_dir=save_dir)

        n = sum(
            len(os.listdir(os.path.join(save_dir, d)))
            for d in VIZ_DIRS.values()
            if os.path.exists(os.path.join(save_dir, d))
        )
        print(f"  Done — {n} figures saved in: {save_dir}")


# =============================================================================
# Paper Figures — auto-select best combinations from averaged CSV
# =============================================================================

def _auto_select(df_orig, df_deriv):
    dfs = []
    if df_orig  is not None: dfs.append(df_orig.assign(input_mode='original'))
    if df_deriv is not None: dfs.append(df_deriv.assign(input_mode='derivative'))
    if not dfs: return {}
    df_all = pd.concat(dfs, ignore_index=True)
    model_r = df_all.groupby('Model')['R'].mean()
    ch_r    = df_all.groupby('Target_CH')['R'].mean()
    task_r  = df_all.groupby('Task')['R'].mean()
    mode_r  = df_all.groupby('input_mode')['R'].mean()
    return {
        'best_model'   : model_r.idxmax(),
        'model_r_mean' : round(model_r.max(), 4),
        'best_ch'      : ch_r.idxmax(),
        'worst_ch'     : ch_r.idxmin(),
        'best_ch_r'    : round(ch_r.max(), 4),
        'worst_ch_r'   : round(ch_r.min(), 4),
        'best_task'    : task_r.idxmax(),
        'task_r_mean'  : round(task_r.max(), 4),
        'best_mode'    : mode_r.idxmax(),
        'mode_r_orig'  : round(mode_r.get('original',   0), 4),
        'mode_r_deriv' : round(mode_r.get('derivative', 0), 4),
    }


def generate_paper_figures(subject_ids=None):
    """
    Auto-select best combinations per subject and generate publication figures.
    Requires averaged CSV files (produced by run_new_scheme / summarize_results).

    Selection criteria:
      - Best model   : highest mean R across all tasks, channels, modes
      - Best channel : highest mean R (for timeseries & scatter)
      - Worst channel: lowest mean R  (contrast)
      - Best task    : highest mean R (representative)
      - Both modes   : always shown for comparison (main contribution)

    Output (~10 figures + selection_log.txt):
      paper_figures/sub{id}/
      ├── fig1_r_bar_comparison_{task}.png  (x3)
      ├── fig2_timeseries_best_{ch}.png
      ├── fig2_timeseries_worst_{ch}.png
      ├── fig3_scatter_best_{ch}.png
      ├── fig3_scatter_worst_{ch}.png
      ├── fig4_distance_vs_R.png
      └── selection_log.txt
    """
    if subject_ids is None:
        subject_ids = SUBJECT_IDS

    for sid in subject_ids:
        print(f"\n{'='*60}")
        print(f"  Paper Figures | Subject: {sid}")
        print(f"{'='*60}")
        viz_dir = os.path.join(PAPER_DIR, f'sub{sid}')
        os.makedirs(viz_dir, exist_ok=True)

        df_o = _load_avg_csv(sid, 'original')
        df_d = _load_avg_csv(sid, 'derivative')
        if df_o is None and df_d is None:
            print(f"  [SKIP] No averaged CSV for sub{sid}")
            continue

        cache_o = load_training_results(sid, 'original')
        cache_d = load_training_results(sid, 'derivative')
        sel     = _auto_select(df_o, df_d)

        log = [
            f"Auto-selection log — Subject {sid}",
            f"{'='*50}",
            f"Best model   : {sel.get('best_model')}  (mean R = {sel.get('model_r_mean')})",
            f"Best channel : {sel.get('best_ch')}  (mean R = {sel.get('best_ch_r')})",
            f"Worst channel: {sel.get('worst_ch')}  (mean R = {sel.get('worst_ch_r')})",
            f"Best task    : {sel.get('best_task')}  (mean R = {sel.get('task_r_mean')})",
            f"Best mode    : {sel.get('best_mode')}",
            f"  original mean R   = {sel.get('mode_r_orig')}",
            f"  derivative mean R = {sel.get('mode_r_deriv')}",
            f"{'='*50}",
            f"Note: both modes always shown (main contribution).",
            f"Timeseries & scatter: {VIZ_SPLIT} (trial 3 as test).",
            f"R bar & distance   : averaged across all 3 LOTO splits.",
        ]
        for line in log: print(f"  {line}")
        with open(os.path.join(viz_dir, 'selection_log.txt'), 'w') as f:
            f.write('\n'.join(log))

        bm = sel.get('best_model', list(MODEL_COLORS.keys())[0])
        bc = sel.get('best_ch',   list(NEW_ESTIMATION_SCHEME.keys())[0])
        wc = sel.get('worst_ch',  list(NEW_ESTIMATION_SCHEME.keys())[-1])
        bt = sel.get('best_task', TASKS[0])

        # Fig 1: R bar comparison (original vs derivative side-by-side)
        channels   = list(NEW_ESTIMATION_SCHEME.keys())
        model_list = list(MODEL_COLORS.keys())
        n_ch, n_m  = len(channels), len(model_list)
        bw, x      = 0.09, np.arange(n_ch)
        for task in TASKS:
            fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
            fig.patch.set_facecolor('#FAFAFA')
            fig.suptitle(
                f'R \u2014 Original vs Derivative | '                f'Task: {task.replace("_"," ")} | Subject: {sid}',
                fontsize=12, fontweight='bold'
            )
            for ax, (mode, df) in zip(axes, [('original',df_o),('derivative',df_d)]):
                if df is None: ax.set_visible(False); continue
                tdf = df[df['Task']==task]
                for i, mn in enumerate(model_list):
                    rv = []
                    for ch in channels:
                        r = tdf[(tdf['Target_CH']==ch)&(tdf['Model']==mn)]['R'].mean()
                        rv.append(r if not np.isnan(r) else 0)
                    off  = (i - n_m/2 + 0.5) * bw
                    bars = ax.bar(x+off, rv, bw, label=mn,
                                  color=MODEL_COLORS[mn], alpha=0.85,
                                  edgecolor='white', linewidth=0.5)
                    for bar, r in zip(bars, rv):
                        if r > 0.05:
                            ax.text(bar.get_x()+bar.get_width()/2,
                                    bar.get_height()+0.01, f'{r:.2f}',
                                    ha='center', va='bottom', fontsize=6, rotation=90)
                ax.axhline(0.7, color='#AAA', linewidth=1.2,
                           linestyle='--', alpha=0.7, label='R = 0.7')
                ax.set_xticks(x); ax.set_xticklabels(channels, fontsize=10)
                ax.set_ylim(0, 1.18); ax.set_ylabel('R', fontsize=11)
                ax.set_xlabel('Target Channel', fontsize=11)
                ax.set_title(MODE_DISPLAY[mode], fontsize=10,
                             fontweight='bold', color=MODE_COLOR[mode])
                ax.set_facecolor('#F8F8F8')
                ax.grid(True, axis='y', linestyle='--', alpha=0.35)
                ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
                ax.legend(fontsize=8, loc='upper right', framealpha=0.88)
            plt.tight_layout()
            _savefig(fig, os.path.join(viz_dir, f"fig1_r_bar_comparison_{task}.png"),
                     dpi=PAPER_DPI)

        # Fig 2: Timeseries best & worst channel
        for ch_lbl, ch in [('best', bc), ('worst', wc)]:
            fig, ax = plt.subplots(figsize=(16, 4))
            fig.patch.set_facecolor('#FAFAFA')
            done, dur = False, None
            for mode, cache in [('original',cache_o),('derivative',cache_d)]:
                if cache is None: continue
                chd = cache.get(bt,{}).get(VIZ_SPLIT,{}).get(ch)
                if chd is None: continue
                y_true  = chd['y_test_orig']
                n_pts   = min(640, len(y_true))
                t_axis  = np.arange(n_pts) / SAMPLING_RATE
                if not done:
                    dur = t_axis[-1]
                    for sec in range(0, int(dur)+1, 2):
                        ax.axvspan(sec, min(sec+1,dur),
                                   color='#E8EAF6', alpha=0.35, zorder=1)
                    ax.plot(t_axis, y_true[:n_pts], color='#1A1A2E',
                            linewidth=1.0, alpha=0.9, label='Actual', zorder=6)
                    done = True
                bm_key = max(chd['predictions'],
                             key=lambda m: compute_metrics(y_true, chd['predictions'][m])['R'])
                ax.plot(t_axis, chd['predictions'][bm_key][:n_pts],
                        color=MODE_COLOR[mode], linewidth=0.9, alpha=0.8,
                        linestyle=MODE_LS[mode],
                        label=f'{MODE_DISPLAY[mode]} ({bm_key})', zorder=5)
            if not done: plt.close(fig); continue
            ax.set_xlim(0, dur); ax.set_xticks(np.arange(0, dur+1, 2))
            ax.set_xlabel('Time (s)', fontsize=11)
            ax.set_ylabel('Amplitude (\u00b5V)', fontsize=11)
            ax.set_title(
                f'Timeseries \u2014 Channel: {ch} ({ch_lbl}) | '                f'Task: {bt.replace("_"," ")} | Subject: {sid}\n'                f'Estimated: best model per mode | Split: trial 3 as test',
                fontsize=11, fontweight='bold'
            )
            ax.set_facecolor('#F4F6FF')
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            ax.legend(fontsize=9, loc='upper right', framealpha=0.88, ncol=3)
            plt.tight_layout()
            _savefig(fig, os.path.join(viz_dir, f"fig2_timeseries_{ch_lbl}_{ch}.png"),
                     dpi=PAPER_DPI)

        # Fig 3: Scatter best & worst channel, best model
        for ch_lbl, ch in [('best', bc), ('worst', wc)]:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            fig.patch.set_facecolor('#FAFAFA')
            fig.suptitle(
                f'Scatter \u2014 Model: {bm} | Channel: {ch} ({ch_lbl}) | '                f'Task: {bt.replace("_"," ")} | Subject: {sid}\n'                f'Split: trial 3 as test',
                fontsize=11, fontweight='bold'
            )
            for ax, (mode, cache) in zip(axes,[('original',cache_o),('derivative',cache_d)]):
                if cache is None: ax.set_visible(False); continue
                chd = cache.get(bt,{}).get(VIZ_SPLIT,{}).get(ch)
                if chd is None: ax.set_visible(False); continue
                y_true = chd['y_test_orig']
                y_pred = chd['predictions'].get(bm)
                if y_pred is None: ax.set_visible(False); continue
                metrics = compute_metrics(y_true, y_pred)
                color   = MODE_COLOR[mode]
                ax.scatter(y_true, y_pred, color=color,
                           alpha=0.25, s=5, edgecolors='none', zorder=3)
                vmin = min(y_true.min(), y_pred.min())
                vmax = max(y_true.max(), y_pred.max())
                ax.plot([vmin,vmax],[vmin,vmax], color='#1A1A2E',
                        linewidth=1.5, linestyle='--', alpha=0.8, zorder=4)
                ax.text(0.04, 0.97,
                        f"R    = {metrics['R']:.4f}\n"
                        f"RMSE = {metrics['RMSE']:.4f}\n"
                        f"MAE  = {metrics['MAE']:.4f}\n"
                        f"NSE  = {metrics['NSE']:.4f}",
                        transform=ax.transAxes, fontsize=9,
                        verticalalignment='top', fontfamily='monospace',
                        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                                  alpha=0.88, edgecolor=color, linewidth=1.2))
                ax.set_xlabel('Actual (\u00b5V)', fontsize=11)
                ax.set_ylabel('Estimated (\u00b5V)', fontsize=11)
                ax.set_title(MODE_DISPLAY[mode], fontsize=10,
                             fontweight='bold', color=color)
                ax.set_facecolor('#F8F8F8')
                ax.grid(True, linestyle='--', alpha=0.35, color='#CCCCCC')
                ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
                ax.set_aspect('equal', adjustable='box')
            plt.tight_layout()
            _savefig(fig, os.path.join(viz_dir, f"fig3_scatter_{ch_lbl}_{ch}.png"),
                     dpi=PAPER_DPI)

        # Fig 4: Distance vs R
        plot_distance_vs_r(sid, df_o, df_d, save_dir=viz_dir)

        n = len([f for f in os.listdir(viz_dir) if f.endswith('.png')])
        print(f"\n  Done — {n} paper figures saved in: {viz_dir}")


## 6. Save / Load Training Results

In [ ]:
def save_training_results(results_dict, subject_id, input_mode,
                           save_dir=OUTPUT_DIR):
    """Save all training results to pickle file."""
    MODE_LABEL = {'original': 'Asli', 'derivative': 'Turunan'}  # file naming kept in Indonesian to match existing cache files
    os.makedirs(save_dir, exist_ok=True)
    fname  = f"training_cache_{MODE_LABEL[input_mode]}_sub{subject_id}.pkl"
    fpath  = os.path.join(save_dir, fname)
    with open(fpath, 'wb') as f:
        pickle.dump(results_dict, f)
    print(f"  [CACHE] Training results saved: {fpath}")
    return fpath


def load_training_results(subject_id, input_mode, save_dir=OUTPUT_DIR):
    """Load training results from pickle file."""
    MODE_LABEL = {'original': 'Asli', 'derivative': 'Turunan'}  # file naming kept in Indonesian to match existing cache files
    fname = f"training_cache_{MODE_LABEL[input_mode]}_sub{subject_id}.pkl"
    fpath = os.path.join(save_dir, fname)
    if not os.path.exists(fpath):
        print(f"  [CACHE] File not found: {fpath}")
        return None
    with open(fpath, 'rb') as f:
        results_dict = pickle.load(f)
    print(f"  [CACHE] Training results loaded: {fpath}")
    return results_dict

## 7. Signal Export

Exports per-trial signal data and rolling Pearson R to CSV for detailed analysis.

In [ ]:
def rolling_pearson(y_true, y_pred, window=SAMPLING_RATE):
    """Rolling Pearson r in a sliding window of `window` samples."""
    n         = len(y_true)
    r_rolling = np.full(n, np.nan)
    for i in range(window - 1, n):
        a = y_true[i - window + 1 : i + 1]
        b = y_pred[i - window + 1 : i + 1]
        if np.std(a) < 1e-10 or np.std(b) < 1e-10:
            r_rolling[i] = np.nan
        else:
            r_rolling[i], _ = pearsonr(a, b)
    return np.round(r_rolling, 6)


def export_signal_comparison(summary_preds, task, subject_id,
                              split_label, estimation_scheme,
                              save_dir=OUTPUT_DIR,
                              rolling_window=SAMPLING_RATE):
    """
    Export CSV per task per subject:
      PART 1 — signal per timepoint: time_s | {CH}_actual | {CH}_{model}_estimated | rolling_r
      PART 2 — global metric summary (R, RMSE, MAE, NSE)
    """
    os.makedirs(save_dir, exist_ok=True)
    n_pts     = next(len(v['actual']) for v in summary_preds.values())
    time_s    = np.round(np.arange(n_pts) / SAMPLING_RATE, 6)
    signal_df = pd.DataFrame({'time_s': time_s})

    for target_ch, ch_data in summary_preds.items():
        y_true = ch_data['actual']
        signal_df[f'{target_ch}_actual'] = np.round(y_true, 6)
        for model_name, y_pred in ch_data.items():
            if model_name == 'actual':
                continue
            r_roll = rolling_pearson(y_true, y_pred, window=rolling_window)
            signal_df[f'{target_ch}_{model_name}_estimated'] = np.round(y_pred, 6)
            signal_df[f'{target_ch}_{model_name}_rolling_r'] = r_roll

    corr_rows = []
    for target_ch, ch_data in summary_preds.items():
        y_true    = ch_data['actual']
        input_chs = estimation_scheme.get(target_ch, [])
        for model_name, y_pred in ch_data.items():
            if model_name == 'actual':
                continue
            m = compute_metrics(y_true, y_pred)
            corr_rows.append({
                'Channel'       : target_ch,
                'Input_Channels': ', '.join(input_chs),
                'Model'         : model_name,
                **m
            })

    corr_df = pd.DataFrame(corr_rows)
    fname   = f"signal_comparison_{split_label}_sub{subject_id}_{task}.csv"
    fpath   = os.path.join(save_dir, fname)

    with open(fpath, 'w', newline='', encoding='utf-8') as f:
        f.write(f"# EEG Signal Comparison | Subject: {subject_id} | "
                f"Task: {task} | Split: {split_label}\n")
        f.write(f"# Sampling rate: {SAMPLING_RATE} Hz | "
                f"Timepoints: {n_pts} ({n_pts/SAMPLING_RATE:.1f} s)\n")
        f.write(f"# PART 1: actual, estimated, rolling_r "
                f"(window {rolling_window} samples = "
                f"{rolling_window/SAMPLING_RATE:.1f} s)\n")
        signal_df.to_csv(f, index=False)
        f.write("\n# PART 2: Metric summary\n")
        corr_df.to_csv(f, index=False)

    print(f"  [EXPORT] Saved: {fpath}")
    return fpath

## 8. Non-Wearable ← Wearable Estimation (LOTO)

Trains all models using Leave-One-Trial-Out cross-validation.
Checkpoints saved after each channel — resume-safe if interrupted.

In [ ]:
def run_new_scheme(subject_id, input_mode='original',
                   data_dir=DATA_DIR, export_csv=True):
    """
    Estimate non-wearable channels from 4 wearable channels.
    Uses full Leave-One-Trial-Out (LOTO): 3 rotations, result = average.

    After training completes, call:
        generate_all(subject_ids=[subject_id])           # all 69 figures
        generate_paper_figures(subject_ids=[subject_id]) # ~10 paper figures

    Parameters:
        subject_id  : int or str
        input_mode  : 'original' | 'derivative'
        data_dir    : path to .mat folder
        export_csv  : True -> export signal comparison CSV (last split)
    """
    MODE_LABEL = {'original': 'Actual', 'derivative': 'Derivative'}
    if input_mode not in MODE_LABEL:
        raise ValueError(f"input_mode must be one of {list(MODE_LABEL.keys())}")

    mode_str = MODE_LABEL[input_mode]

    print(f"\n{'='*60}")
    print(f"  Non-Wearable Estimation | Subject: {subject_id}")
    print(f"  Input mode : {input_mode} (LOTO — 3 rotations)")
    print(f"{'='*60}")

    all_results = []
    cache_data  = {}

    existing = load_training_results(subject_id, input_mode)
    if existing:
        cache_data = existing
        print("  [RESUME] Cache found, resuming from checkpoint...")

    for task in TASKS:
        print(f"\n[Task: {task}]")
        if task not in cache_data:
            cache_data[task] = {}

        trial_data = []
        for trial in range(1, N_TRIALS + 1):
            fp = os.path.join(data_dir,
                              f"{task}_sub_{subject_id}_trial{trial}.mat")
            if not os.path.exists(fp):
                print(f"  [WARNING] Not found: {fp}")
                trial_data.append(None)
                continue
            df = load_mat(fp)
            trial_data.append(df)
            print(f"  Loaded: {os.path.basename(fp)} -> {df.shape}")

        if any(t is None for t in trial_data):
            print(f"  [SKIP] Incomplete data for task {task}")
            continue

        for test_idx in range(N_TRIALS):
            split_key   = f'split_{test_idx}'
            split_label = f"LOTO_trial{test_idx + 1}_as_test_{mode_str}"
            is_last     = (test_idx == N_TRIALS - 1)

            train_idx = [i for i in range(N_TRIALS) if i != test_idx]
            df_train  = pd.concat([trial_data[i] for i in train_idx],
                                   axis=0).reset_index(drop=True)
            df_test   = trial_data[test_idx].reset_index(drop=True)

            print(f"\n  --- Split: Trial {test_idx+1} as test ---")
            print(f"  Train: {len(df_train)} rows | Test: {len(df_test)} rows")

            if split_key not in cache_data[task]:
                cache_data[task][split_key] = {}

            X_tr_feat, feat_cols = expand_features(df_train, WEARABLE_CH,
                                                    mode=input_mode)
            X_te_feat, _         = expand_features(df_test,  WEARABLE_CH,
                                                    mode=input_mode)
            print(f"  Features: {len(feat_cols)} -> {feat_cols}")

            summary_preds = {}

            for target_ch, input_chs in NEW_ESTIMATION_SCHEME.items():

                if (target_ch in cache_data[task][split_key] and
                        'metrics_test' in cache_data[task][split_key][target_ch]):
                    print(f"  [SKIP] {target_ch} split {test_idx+1} already done.")
                    ch_cache = cache_data[task][split_key][target_ch]
                    summary_preds[target_ch] = {
                        'actual': ch_cache['y_test_orig'],
                        **ch_cache['predictions']
                    }
                    for mn, metrics in ch_cache['metrics_test'].items():
                        all_results.append({
                            'Subject': subject_id, 'Task': task,
                            'Target_CH': target_ch,
                            'Input_CH': ', '.join(input_chs),
                            'Input_Mode': input_mode,
                            'N_Features': len(feat_cols),
                            'Model': mn, 'Split': split_label,
                            'Test_Trial': test_idx + 1,
                            'ANN_Best_Params': str(ch_cache['ann_best_params'])
                                              if mn == 'ANN_MLP' else '-',
                            **metrics
                        })
                    continue

                print(f"\n  Estimating [{target_ch}] <- {len(feat_cols)} features")

                X_tr = X_tr_feat.values
                X_te = X_te_feat.values
                y_tr_raw = df_train[target_ch].values.reshape(-1, 1)
                y_te_raw = df_test[target_ch].values.reshape(-1, 1)

                X_tr_n, X_te_n, _       = normalize_data(X_tr, X_te)
                y_tr_n, y_te_n, sc_y    = normalize_data(y_tr_raw, y_te_raw)
                y_tr_flat               = y_tr_n.ravel()
                y_te_orig               = sc_y.inverse_transform(y_te_n).ravel()
                y_tr_orig               = sc_y.inverse_transform(y_tr_n).ravel()

                predictions_dict = {}
                train_preds_dict = {}
                summary_preds[target_ch] = {'actual': y_te_orig}

                models = build_models()
                print(f"    [TUNING] ANN for {target_ch} [split {test_idx+1}]...")
                tuned_ann, best_params, lr_history = train_keras_ann(
                    X_tr_n, y_tr_flat, X_te_n, y_te_orig, sc_y,
                    input_dim=X_tr_n.shape[1]
                )
                models['ANN_MLP'] = tuned_ann
                print(f"    [TUNING] Best params: {best_params}")

                for mn, model in models.items():
                    try:
                        if mn != 'ANN_MLP':
                            model.fit(X_tr_n, y_tr_flat)

                        raw_tr = (model.predict(X_tr_n, verbose=0)
                                  if mn == 'ANN_MLP'
                                  else model.predict(X_tr_n).reshape(-1,1))
                        raw_te = (model.predict(X_te_n, verbose=0)
                                  if mn == 'ANN_MLP'
                                  else model.predict(X_te_n).reshape(-1,1))

                        train_preds_dict[mn] = sc_y.inverse_transform(
                            raw_tr.reshape(-1,1)).ravel()
                        y_pred = sc_y.inverse_transform(
                            raw_te.reshape(-1,1)).ravel()

                        predictions_dict[mn]         = y_pred
                        summary_preds[target_ch][mn] = y_pred

                        metrics = compute_metrics(y_te_orig, y_pred)
                        all_results.append({
                            'Subject': subject_id, 'Task': task,
                            'Target_CH': target_ch,
                            'Input_CH': ', '.join(input_chs),
                            'Input_Mode': input_mode,
                            'N_Features': len(feat_cols),
                            'Model': mn, 'Split': split_label,
                            'Test_Trial': test_idx + 1,
                            'ANN_Best_Params': str(best_params)
                                              if mn == 'ANN_MLP' else '-',
                            **metrics
                        })
                        print(f"    {mn:20s} -> R={metrics['R']:.4f} | "
                              f"RMSE={metrics['RMSE']:.6f} | "
                              f"NSE={metrics['NSE']:.4f}")

                    except Exception as e:
                        print(f"    [ERROR] {mn}: {e}")

                cache_data[task][split_key][target_ch] = {
                    'y_test_orig'    : y_te_orig,
                    'y_train_orig'   : y_tr_orig,
                    'predictions'    : predictions_dict,
                    'train_preds'    : train_preds_dict,
                    'metrics_test'   : {m: compute_metrics(y_te_orig, p)
                                        for m, p in predictions_dict.items()},
                    'metrics_train'  : {m: compute_metrics(y_tr_orig, p)
                                        for m, p in train_preds_dict.items()},
                    'ann_best_params': best_params,
                    'ann_lr_history' : lr_history,
                }
                save_training_results(cache_data, subject_id, input_mode)
                print(f"    [CHECKPOINT] task={task}, ch={target_ch}, "
                      f"split={test_idx+1}")

            # Export signal CSV for last split
            if is_last and export_csv and summary_preds:
                export_signal_comparison(
                    summary_preds=summary_preds,
                    task=task,
                    subject_id=subject_id,
                    split_label=split_label,
                    estimation_scheme=NEW_ESTIMATION_SCHEME,
                    save_dir=os.path.join(OUTPUT_DIR,
                                          f'signal_comparison_sub{subject_id}'),
                )

    results_df = pd.DataFrame(all_results)
    per_split_path = os.path.join(
        OUTPUT_DIR, f"results_{mode_str}_sub{subject_id}_per_split.csv")
    results_df.to_csv(per_split_path, index=False)

    loto_avg = (
        results_df
        .groupby(['Subject', 'Task', 'Target_CH', 'Input_CH',
                  'Input_Mode', 'Model'])
        [['R', 'RMSE', 'MAE', 'NSE']]
        .mean().round(4).reset_index()
    )
    loto_avg['Split'] = 'LOTO_averaged'
    avg_path = os.path.join(
        OUTPUT_DIR, f"results_{mode_str}_sub{subject_id}_averaged.csv")
    loto_avg.to_csv(avg_path, index=False)
    save_training_results(cache_data, subject_id, input_mode)

    print(f"\n  Per-split CSV : {per_split_path}")
    print(f"  Averaged CSV  : {avg_path}")
    print(f"\n  [LOTO summary] Mean R across 3 splits:")
    print(loto_avg.groupby(['Task','Target_CH','Model'])['R']
          .mean().round(4).to_string())

    return loto_avg


## 9. Multi-Subject Summary

In [ ]:
def summarize_results(results_list, label='summary'):
    """
    Average metrics across all subjects, grouped by Task/Channel/Mode/Model.
    Save to CSV and display summary.
    """
    df_all = pd.concat(results_list, axis=0).reset_index(drop=True)

    group_cols = ['Task', 'Target_CH', 'Input_Mode', 'Model']
    group_cols = [c for c in group_cols if c in df_all.columns]

    summary_df = (
        df_all
        .groupby(group_cols)[['R', 'RMSE', 'MAE', 'NSE']]
        .mean().round(4).reset_index()
    )
    out_path = os.path.join(OUTPUT_DIR, f"summary_{label}_all_subjects.csv")
    summary_df.to_csv(out_path, index=False)
    print(f"Summary saved: {out_path}")
    return summary_df

## 10. Execution

Adjust `SUBJECT_IDS` and `MODES_TO_RUN` before running.
After training completes, visualizations are generated automatically.

In [ ]:
import time  # tambahkan di baris paling atas cell 20

# =============================================================================
# EXECUTION CONFIG — adjust before running
# =============================================================================
SUBJECT_IDS  = [30,31,32,33,34,35,36,37,38,39,40]                        # subjects to process
MODES_TO_RUN = ['original', 'derivative']  # input modes

# =============================================================================
# TRAINING — per subject per mode
# =============================================================================
all_results_per_mode  = {}
loto_avg_per_subject  = {}

for sid in SUBJECT_IDS:
    loto_avg_per_subject[sid] = {}
    start_subject = time.time()

    for mode in MODES_TO_RUN:
        print(f"\n{'#'*60}")
        print(f"# SUBJECT: {sid} | MODE: {mode.upper()}")
        print(f"{'#'*60}")
        start_mode = time.time()

        res = run_new_scheme(
            subject_id=sid,
            input_mode=mode,
            data_dir=DATA_DIR,
            export_csv=True
        )

        elapsed_mode = time.time() - start_mode
        print(f"\n[TIMING] Subject {sid} | Mode {mode}: {elapsed_mode:.1f}s ({elapsed_mode/60:.1f} min)")
        loto_avg_per_subject[sid][mode] = res

        if mode not in all_results_per_mode:
            all_results_per_mode[mode] = []
        all_results_per_mode[mode].append(res)

    # --- Visualizations (after both modes done for this subject) ---
    print(f"\n{'#'*60}")
    print(f"# VISUALIZATIONS | Subject: {sid}")
    print(f"{'#'*60}")
    elapsed_subject = time.time() - start_subject
    print(f"\n[TIMING] Subject {sid} TOTAL (training only): {elapsed_subject:.1f}s ({elapsed_subject/60:.1f} min)")
    generate_all(subject_ids=[sid])
    generate_paper_figures(subject_ids=[sid])

# =============================================================================
# MULTI-SUBJECT SUMMARY
# =============================================================================
for mode in MODES_TO_RUN:
    if all_results_per_mode.get(mode):
        summary = summarize_results(all_results_per_mode[mode], label=mode)
        print(f"\n--- Summary Mode: {mode} ---")
        print(summary[['Task','Target_CH','Model','R','RMSE','NSE']]
              .to_string(index=False))



############################################################
# SUBJECT: 30 | MODE: ORIGINAL
############################################################

  Non-Wearable Estimation | Subject: 30
  Input mode : original (LOTO — 3 rotations)
  [CACHE] Training results loaded: /content/drive/MyDrive/Thesis/channel_estimation/training_cache_Asli_sub30.pkl
  [RESUME] Cache found, resuming from checkpoint...

[Task: Arithmetic]
  Loaded: Arithmetic_sub_30_trial1.mat -> (3200, 32)
  Loaded: Arithmetic_sub_30_trial2.mat -> (3200, 32)
  Loaded: Arithmetic_sub_30_trial3.mat -> (3200, 32)

  --- Split: Trial 1 as test ---
  Train: 6400 rows | Test: 3200 rows
  Features: 4 -> ['Fp1', 'Fp2', 'F7', 'F8']
  [SKIP] FC2 split 1 already done.
  [SKIP] FZ split 1 already done.
  [SKIP] CP2 split 1 already done.
  [SKIP] CZ split 1 already done.
  [SKIP] P4 split 1 already done.

  --- Split: Trial 2 as test ---
  Train: 6400 rows | Test: 3200 rows
  Features: 4 -> ['Fp1', 'Fp2', 'F7', 'F8']
  [SKIP] FC2 

  Saved: plots_sub30/r_bar/Actual_Mirror_image.png
  [4/4] Overfitting detection...


  Saved: plots_sub30/overfitting/Actual.png
  [CACHE] Training results loaded: /content/drive/MyDrive/Thesis/channel_estimation/training_cache_Turunan_sub30.pkl

  Mode: derivative
  [1/4] Timeseries...
  Saved: plots_sub30/timeseries/Derivative_Arithmetic_FC2.png
  Saved: plots_sub30/timeseries/Derivative_Arithmetic_FZ.png
  Saved: plots_sub30/timeseries/Derivative_Arithmetic_CP2.png
  Saved: plots_sub30/timeseries/Derivative_Arithmetic_CZ.png
  Saved: plots_sub30/timeseries/Derivative_Arithmetic_P4.png
  Saved: plots_sub30/timeseries/Derivative_Stroop_FC2.png
  Saved: plots_sub30/timeseries/Derivative_Stroop_FZ.png
  Saved: plots_sub30/timeseries/Derivative_Stroop_CP2.png
  Saved: plots_sub30/timeseries/Derivative_Stroop_CZ.png
  Saved: plots_sub30/timeseries/Derivative_Stroop_P4.png
  Saved: plots_sub30/timeseries/Derivative_Mirror_image_FC2.png
  Saved: plots_sub30/timeseries/Derivative_Mirror_image_FZ.png
  Saved: plots_sub30/timeseries/Derivative_Mirror_image_CP2.png
  Saved: plo

  Saved: plots_sub30/r_bar/Derivative_Mirror_image.png
  [4/4] Overfitting detection...


  Saved: plots_sub30/overfitting/Derivative.png
  [5/5] Distance vs R...
  Saved: plots_sub30/distance/distance_vs_R.png
  Done — 69 figures saved in: /content/drive/MyDrive/Thesis/channel_estimation/plots_sub30

  Paper Figures | Subject: 30
  [CACHE] Training results loaded: /content/drive/MyDrive/Thesis/channel_estimation/training_cache_Asli_sub30.pkl
  [CACHE] Training results loaded: /content/drive/MyDrive/Thesis/channel_estimation/training_cache_Turunan_sub30.pkl
  Auto-selection log — Subject 30
  Best model   : ANN_MLP  (mean R = 0.7209)
  Best channel : FZ  (mean R = 0.8861)
  Worst channel: CZ  (mean R = 0.6242)
  Best task    : Mirror_image  (mean R = 0.7205)
  Best mode    : derivative
    original mean R   = 0.7089
    derivative mean R = 0.7128
  Note: both modes always shown (main contribution).
  Timeseries & scatter: split_2 (trial 3 as test).
  R bar & distance   : averaged across all 3 LOTO splits.
  Saved: paper_figures/sub30/fig1_r_bar_comparison_Arithmetic.png
  S

In [ ]:
# Top 3 kombinasi per channel berdasarkan frekuensi
top3_per_channel = {}

for ch in ['CP2', 'CZ', 'FC2', 'FZ', 'P4']:
    grp = df_params[df_params['Channel'] == ch]
    top3 = (grp.groupby(['arch','lr'])
               .size()
               .reset_index(name='count')
               .sort_values('count', ascending=False)
               .head(3))
    top3_per_channel[ch] = [
        {'architecture': eval(row['arch']),
         'learning_rate': row['lr']}
        for _, row in top3.iterrows()
    ]
    print(f"{ch}:")
    for p in top3_per_channel[ch]:
        print(f"  arch={p['architecture']} lr={p['learning_rate']}")
    print()

# Simpan
import json
with open(os.path.join(OUTPUT_DIR, 'top3_ann_params.json'), 'w') as f:
    json.dump(
        {ch: [{'architecture': list(p['architecture']),
               'learning_rate': p['learning_rate']}
              for p in combos]
         for ch, combos in top3_per_channel.items()},
        f, indent=2
    )
print("Saved: top3_ann_params.json")

In [ ]:
# Perlu data R per kombinasi — ambil dari per_split CSV
records_r = []

for sub_id in range(1, 11):
    for mode_label in ['Asli', 'Turunan']:
        cache_path = os.path.join(
            OUTPUT_DIR,
            f"training_cache_{mode_label}_sub{sub_id}.pkl")
        if not os.path.exists(cache_path):
            continue
        with open(cache_path, "rb") as f:
            cache = pickle.load(f)
        for task, splits in cache.items():
            for split_key, channels in splits.items():
                for ch, result in channels.items():
                    if 'ann_best_params' not in result:
                        continue
                    p = result['ann_best_params']
                    r = result['metrics_test']['ANN_MLP']['R']
                    records_r.append({
                        'Channel' : ch,
                        'arch'    : str(p['architecture']),
                        'lr'      : p['learning_rate'],
                        'R'       : r
                    })

df_r = pd.DataFrame(records_r)

print("=== Mean R per kombinasi per channel ===")
for ch in ['CP2', 'CZ', 'FC2', 'FZ', 'P4']:
    grp = df_r[df_r['Channel'] == ch]
    summary = (grp.groupby(['arch','lr'])['R']
                  .agg(['mean','std','count'])
                  .reset_index()
                  .sort_values('mean', ascending=False))
    print(f"\n{ch}:")
    print(summary.round(4).to_string(index=False))

## 11. Mode Comparison

Pivot table comparing mean R between `original` and `derivative` modes.
Run after execution cell completes.

In [ ]:
# Reconstruct all_results_per_mode from CSV if kernel was restarted
if 'all_results_per_mode' not in dir() or not all_results_per_mode:
    all_results_per_mode = {}
    for mode in ['original', 'derivative']:
        label = {'original': 'Actual', 'derivative': 'Derivative'}[mode]
        results = []
        for sid in SUBJECT_IDS:
            csv_path = os.path.join(
                OUTPUT_DIR, f"results_{label}_sub{sid}_averaged.csv")
            if os.path.exists(csv_path):
                results.append(pd.read_csv(csv_path))
        if results:
            all_results_per_mode[mode] = results

print("\n" + "="*60)
print("  MEAN R COMPARISON: ORIGINAL vs DERIVATIVE")
print("="*60)

all_dfs = []
for mode, res_list in all_results_per_mode.items():
    df = pd.concat(res_list).copy()
    df['Input_Mode'] = mode
    all_dfs.append(df)

df_compare = pd.concat(all_dfs)
pivot = (df_compare
         .groupby(['Target_CH', 'Model', 'Input_Mode'])['R']
         .mean().round(4)
         .unstack('Input_Mode')
         .reset_index())
print(pivot.to_string(index=False))


## 12. Generate Visualizations from Checkpoint

Use these functions to regenerate visualizations after training,
without rerunning the full training loop.

In [ ]:
def regenerate_visualizations(subject_ids=None, modes=None):
    """
    Regenerate all figures from existing cache pkl files.
    Use when training is done but plots need to be recreated.
    """
    if subject_ids is None: subject_ids = SUBJECT_IDS
    if modes is None: modes = ['original', 'derivative']

    generate_all(subject_ids=subject_ids)
    generate_paper_figures(subject_ids=subject_ids)


def regenerate_paper_only(subject_ids=None):
    """Regenerate only paper figures from existing averaged CSVs."""
    if subject_ids is None: subject_ids = SUBJECT_IDS
    generate_paper_figures(subject_ids=subject_ids)


# --- Example usage ---
# regenerate_visualizations(subject_ids=[1,2,3,4,5,6,7,8,9,10])
regenerate_paper_only(subject_ids=[1,2,3,4,5,6,7,8,9,10])


In [ ]:
# =============================================================================
# MEDIAN SUBJECT FINDER
# Cari subjek representatif berdasarkan mean R across semua channel, model, task
# Jalankan setelah semua averaged CSV tersedia
# =============================================================================

import pandas as pd
import numpy as np
import glob
import os

# ── 1. Load semua averaged CSV dari 10 subjek (actual signal) ─────────────
avg_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "results_Actual_sub*_averaged.csv")))

if not avg_files:
    print("[ERROR] Tidak ada file results_Actual_sub*_averaged.csv di OUTPUT_DIR")
    print(f"OUTPUT_DIR saat ini: {OUTPUT_DIR}")
else:
    print(f"Ditemukan {len(avg_files)} file:")
    for f in avg_files:
        print(f"  {os.path.basename(f)}")

    df_all = pd.concat([pd.read_csv(f) for f in avg_files], ignore_index=True)

    # ── 2. Hitung mean R per subjek across semua kombinasi ─────────────────
    per_subject = (
        df_all.groupby("Subject")["R"]
        .mean()
        .reset_index()
        .rename(columns={"R": "mean_R"})
        .sort_values("mean_R")
        .reset_index(drop=True)
    )

    print("\nRanking subjek berdasarkan mean R (ascending):")
    print(per_subject.to_string(index=True))

    # ── 3. Pilih subjek median (posisi tengah dari 10 subjek = index 4 atau 5)
    median_idx     = len(per_subject) // 2        # index 5 dari 0-9
    median_subject = int(per_subject.iloc[median_idx]["Subject"])
    median_r       = per_subject.iloc[median_idx]["mean_R"]

    print(f"\nSubjek median (index {median_idx}): Subject {median_subject}")
    print(f"Mean R: {median_r:.4f}")

    # ── 4. Tampilkan juga subjek terdekat ke overall mean ──────────────────
    overall_mean = per_subject["mean_R"].mean()
    per_subject["dist_to_mean"] = abs(per_subject["mean_R"] - overall_mean)
    closest = per_subject.loc[per_subject["dist_to_mean"].idxmin()]

    print(f"\nOverall mean R (semua subjek): {overall_mean:.4f}")
    print(f"Subjek paling dekat ke mean  : Subject {int(closest['Subject'])} "
          f"(mean R = {closest['mean_R']:.4f}, "
          f"selisih = {closest['dist_to_mean']:.4f})")

    # ── 5. Cek best dan worst channel per subjek ───────────────────────────
    ch_per_sub = (
        df_all.groupby(["Subject", "Target_CH"])["R"]
        .mean()
        .reset_index()
    )

    best_ch_per_sub  = ch_per_sub.loc[ch_per_sub.groupby("Subject")["R"].idxmax()]
    worst_ch_per_sub = ch_per_sub.loc[ch_per_sub.groupby("Subject")["R"].idxmin()]

    print("\nBest channel per subjek:")
    print(best_ch_per_sub[["Subject", "Target_CH", "R"]].to_string(index=False))

    print("\nWorst channel per subjek:")
    print(worst_ch_per_sub[["Subject", "Target_CH", "R"]].to_string(index=False))

    print("\nFrekuensi best channel:")
    print(best_ch_per_sub["Target_CH"].value_counts().to_string())

    print("\nFrekuensi worst channel:")
    print(worst_ch_per_sub["Target_CH"].value_counts().to_string())

    # ── 6. Cek best task per subjek ────────────────────────────────────────
    task_per_sub = (
        df_all.groupby(["Subject", "Task"])["R"]
        .mean()
        .reset_index()
    )
    best_task_per_sub = task_per_sub.loc[task_per_sub.groupby("Subject")["R"].idxmax()]

    print("\nBest task per subjek:")
    print(best_task_per_sub[["Subject", "Task", "R"]].to_string(index=False))

    print("\nFrekuensi best task:")
    print(best_task_per_sub["Task"].value_counts().to_string())

    # ── 7. Summary untuk pemilihan figure paper ────────────────────────────
    print("\n" + "="*60)
    print("REKOMENDASI UNTUK FIGURE PAPER:")
    print("="*60)
    print(f"Subjek representatif (median)  : Subject {median_subject}")
    print(f"Subjek paling dekat mean       : Subject {int(closest['Subject'])}")
    print(f"Best channel (konsisten)       : {best_ch_per_sub['Target_CH'].value_counts().idxmax()}")
    print(f"Worst channel (konsisten)      : {worst_ch_per_sub['Target_CH'].value_counts().idxmax()}")
    print(f"Best task (dominan)            : {best_task_per_sub['Task'].value_counts().idxmax()}")
    print("="*60)

## 13. EEG Electrode Position Map

Visualizes the 32-channel layout of SAM-40, highlighting wearable input
channels (red squares) and non-wearable target channels (green diamonds).

In [ ]:
import matplotlib.patches as _mpatches

_coords_raw = {
    'Cz' : (0,      0.00056), 'Fz'  : (0,      0.275),
    'Fp1': (-18,    0.53644), 'F7'  : (-54,    0.53628),
    'F3' : (-39,    0.35611), 'FC1' : (-44.9,  0.19222),
    'C3' : (-89.9,  0.27444), 'FC5' : (-69,    0.41889),
    'FT9': (-72,    0.66111), 'T7'  : (-90,    0.536),
    'CP5': (-111,   0.41833), 'CP1' : (-135,   0.19167),
    'P3' : (-141,   0.35556), 'P7'  : (-126,   0.53572),
    'PO9': (-145,   0.66056), 'O1'  : (-162,   0.53556),
    'Pz' : (180,    0.27389), 'Oz'  : (180,    0.5355),
    'O2' : (162,    0.53556), 'PO10': (145,    0.66056),
    'P8' : (126,    0.53572), 'P4'  : (141,    0.35556),
    'CP2': (135,    0.19167), 'CP6' : (111,    0.41833),
    'T8' : (90,     0.536),   'FT10': (72,     0.66111),
    'FC6': (69,     0.41889), 'C4'  : (89.9,   0.27444),
    'FC2': (44.9,   0.19222), 'F4'  : (39,     0.35611),
    'F8' : (54,     0.53628), 'Fp2' : (18,     0.53644),
}

def _polar_to_cart(theta_deg, rho):
    theta = np.deg2rad(theta_deg)
    return rho * np.sin(theta), rho * np.cos(theta)

_WEARABLE_LIST = ['Fp1', 'Fp2', 'F7', 'F8']
_TARGETS_LIST  = ['FC2', 'Fz', 'CP2', 'Cz', 'P4']

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_aspect('equal'); ax.axis('off')

ax.add_patch(plt.Circle((0,0), 1.0, color='white', ec='black', lw=2, zorder=0))
for sign in [-1, 1]:
    ax.add_patch(_mpatches.Ellipse(
        (sign*1.02, 0), width=0.06, height=0.15,
        color='white', ec='black', lw=2, zorder=1))
ax.plot([-.07, 0, .07], [0.97, 1.10, 0.97], 'k-', lw=2, zorder=2)

for ch, (theta, rho) in _coords_raw.items():
    x, y = _polar_to_cart(theta, rho)
    if ch in _WEARABLE_LIST:
        color, marker, size = '#E63946', 's', 120
        fc, fs, fw = '#A31621', 8.5, 'bold'
    elif ch in _TARGETS_LIST:
        color, marker, size = '#2A9D8F', 'D', 120
        fc, fs, fw = '#0F6E56', 8.5, 'bold'
    else:
        color, marker, size = '#AAAAAA', 'o', 60
        fc, fs, fw = '#555555', 7.5, 'normal'
    ax.scatter(x, y, c=color, s=size, marker=marker,
               zorder=5, edgecolors='white', linewidths=0.5)
    ha = 'left' if x >= 0 else 'right'
    ox = 0.04 if x >= 0 else -0.04
    ax.text(x+ox, y+0.05, ch, fontsize=fs, ha=ha, va='bottom',
            color=fc, fontweight=fw, zorder=6)

ax.legend(handles=[
    _mpatches.Patch(facecolor='#E63946', label='Wearable input (Fp1, Fp2, F7, F8)'),
    _mpatches.Patch(facecolor='#2A9D8F', label='Target channel (FC2, Fz, CP2, Cz, P4)'),
    _mpatches.Patch(facecolor='#AAAAAA', label='Other channels'),
], loc='lower center', bbox_to_anchor=(0.5, -0.08), ncol=1,
   fontsize=9, frameon=True, edgecolor='#CCCCCC')

ax.set_xlim(-1.25, 1.25); ax.set_ylim(-1.25, 1.35)
ax.set_title('EEG Electrode Positions (Top View)\nSAM-40 Dataset — 32 Channels',
             fontsize=11, fontweight='bold', pad=10)
plt.tight_layout()

out = os.path.join(OUTPUT_DIR, 'eeg_channel_positions.png')
plt.savefig(out, dpi=1000, bbox_inches='tight', facecolor='white')
plt.savefig(out.replace('.png', '.pdf'), bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: {out}")


## 14. Spatial Distance Analysis

Computes Euclidean distances from each target channel to the 4 wearable channels
and correlates them with estimation accuracy. Used to interpret why some channels
are estimated better than others.

In [ ]:
# Cartesian coordinates from polar
_cart = {ch: _polar_to_cart(*v) for ch, v in _coords_raw.items()}

print("Euclidean distances — target to each wearable channel:")
print("-" * 50)
for target in ['FC2', 'Fz', 'CP2', 'CZ', 'P4']:
    tx, ty    = _cart[target]
    distances = []
    for w in ['Fp1', 'Fp2', 'F7', 'F8']:
        wx, wy = _cart[w]
        d = np.sqrt((tx-wx)**2 + (ty-wy)**2)
        distances.append(d)
        print(f"  {target} <-> {w}: {d:.4f}")
    print(f"  {target} mean distance: {np.mean(distances):.4f}\n")

# =============================================================================
# Distance vs Estimation Accuracy (static version with hardcoded R values)
# Update mean_r_ann after training to get the real plot.
# For the data-driven version, use plot_distance_vs_r() from Section 5.
# =============================================================================
from scipy.stats import pearsonr as _pr

channels  = ['CP2', 'Cz', 'FC2', 'Fz', 'P4']
mean_dist = [MEAN_DIST[ch] for ch in channels]

# Replace these with actual values from averaged CSV after training:
# df = pd.read_csv(os.path.join(OUTPUT_DIR, 'results_Actual_sub1_averaged.csv'))
# mean_r_ann = df[df['Model']=='ANN_MLP'].groupby('Target_CH')['R'].mean().reindex(channels).values
mean_r_ann = [0.5721, 0.5715, 0.7342, 0.9239, 0.5625]   # placeholder values

r_corr, p_val = _pr(mean_dist, mean_r_ann)
coeffs  = np.polyfit(mean_dist, mean_r_ann, 1)
x_line  = np.linspace(min(mean_dist)-.03, max(mean_dist)+.03, 100)
d_thr   = (0.7 - coeffs[1]) / coeffs[0]

import matplotlib.patheffects as _pe2
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(mean_dist, mean_r_ann, s=120, color='#185FA5', zorder=5,
           edgecolors='white', linewidths=0.8)
ax.plot(x_line, np.poly1d(coeffs)(x_line), color='#D85A30',
        linewidth=1.5, linestyle='--', zorder=3, label='Linear fit')
ax.axhline(0.7, color='#888', linewidth=1.0, linestyle=':',
           zorder=2, label='R = 0.70 threshold')
ax.axvline(d_thr, color='#888', linewidth=1.0, linestyle=':', zorder=2)

_offsets = {
    'Fz': (-0.02,0.020), 'FC2':(0.01,0.018),
    'Cz':(0.01,-0.028), 'CP2':(-0.02,0.020), 'P4':(0.01,0.018)
}
for i, ch in enumerate(channels):
    ox, oy = _offsets[ch]
    ax.annotate(ch, xy=(mean_dist[i], mean_r_ann[i]),
                xytext=(mean_dist[i]+ox, mean_r_ann[i]+oy),
                fontsize=10, fontweight='bold', color='#185FA5',
                ha='center', va='bottom',
                path_effects=[_pe2.withStroke(linewidth=2, foreground='white')])

ax.text(0.97, 0.97,
        f'R = {r_corr:.4f}\np = {p_val:.4f}',
        transform=ax.transAxes, fontsize=9, ha='right', va='top',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor='#CCC', linewidth=0.8),
        fontfamily='monospace')
ax.text(d_thr+0.01, 0.46, f'd = {d_thr:.2f}', fontsize=8,
        color='#888', va='bottom')

ax.set_xlabel('Mean Distance to Wearable Channels (polar units)', fontsize=11)
ax.set_ylabel('Mean R (ANN_MLP)', fontsize=11)
ax.set_title('Spatial Distance vs. Estimation Accuracy\n'             'ANN_MLP — Averaged Across Tasks and LOTO Splits',
             fontsize=11, fontweight='bold')
ax.set_xlim(0.28, 0.87); ax.set_ylim(0.44, 1.00)
ax.grid(True, linestyle='--', alpha=0.35, color='#CCC')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(fontsize=9, loc='upper right', framealpha=0.9)
plt.tight_layout()

out = os.path.join(OUTPUT_DIR, 'spatial_distance_vs_R.png')
plt.savefig(out, dpi=1000, bbox_inches='tight', facecolor='white')
plt.savefig(out.replace('.png','.pdf'), bbox_inches='tight', facecolor='white')
plt.show()
print(f"Pearson r = {r_corr:.4f}, p = {p_val:.4f}")
print(f"Regression: R = {coeffs[0]:.4f} * dist + {coeffs[1]:.4f}")
print(f"Threshold distance (R=0.70): {d_thr:.4f}")


In [ ]:
import glob

# Load semua averaged CSV
files_o = sorted(glob.glob(os.path.join(OUTPUT_DIR, "results_Actual_sub*_averaged.csv")))
files_d = sorted(glob.glob(os.path.join(OUTPUT_DIR, "results_Derivative_sub*_averaged.csv")))

df_o_all = pd.concat([pd.read_csv(f) for f in files_o], ignore_index=True)
df_d_all = pd.concat([pd.read_csv(f) for f in files_d], ignore_index=True)

# Rata-ratakan across 10 subjek
df_o_mean = df_o_all.groupby("Target_CH")["R"].mean().reset_index()
df_d_mean = df_d_all.groupby("Target_CH")["R"].mean().reset_index()

# Plot dengan fungsi yang sudah ada, pakai "N=10" sebagai subject_id
plot_distance_vs_r(
    subject_id="N=10 (mean)",
    df_orig=df_o_all,
    df_deriv=df_d_all,
    save_dir=os.path.join(OUTPUT_DIR, "paper_figures/population")
)

In [ ]:
# =============================================================================
# distance_r_correlation.py
#
# Computes mean Pearson R of ANN_MLP per target channel,
# averaged across all subjects and all tasks, for each input mode.
# Then correlates with mean spatial distance and plots the result.
#
# Output:
#   - Console: mean R per channel, Pearson r, p-value, regression
#   - Figure : dual Y-axis bar + line chart (saved to OUTPUT_DIR)
#   - CSV    : summary table (saved to OUTPUT_DIR)
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# =============================================================================
# CONFIG — adjust before running
# =============================================================================
OUTPUT_DIR  = '/content/drive/MyDrive/Thesis/channel_estimation'
SUBJECT_IDS = [1]    # extend to all subjects, e.g. [1, 2, ..., 10]
MODEL_NAME  = 'ANN_MLP'

# Channel order for the plot (sorted by distance ascending)
CHANNELS = ['FZ', 'FC2', 'CZ', 'CP2', 'P4']

# Pre-computed mean Euclidean distance from each target channel
# to the 4 wearable channels (Fp1, Fp2, F7, F8) in polar coordinate units
MEAN_DIST = {
    'FZ' : 0.3617,
    'FC2': 0.4501,
    'CZ' : 0.5360,
    'CP2': 0.6564,
    'P4' : 0.7946,
}


# =============================================================================
# HELPERS
# =============================================================================

def load_avg_csv(subject_id, input_mode):
    label = {'original': 'Actual', 'derivative': 'Derivative'}[input_mode]
    path  = os.path.join(OUTPUT_DIR,
                         f"results_{label}_sub{subject_id}_averaged.csv")
    if not os.path.exists(path):
        print(f"  [WARN] Not found: {path}")
        return None
    return pd.read_csv(path)


# =============================================================================
# MAIN ANALYSIS
# =============================================================================

def compute_distance_r_correlation(subject_ids=None, model_name=MODEL_NAME,
                                   save_fig=True):
    """
    Compute mean Pearson R of ANN_MLP per channel, averaged across all
    subjects and tasks, for each input mode. Then correlate with distance.

    Parameters:
        subject_ids : list of subject IDs to include
        model_name  : model to analyze (default: ANN_MLP)
        save_fig    : True -> save figure to OUTPUT_DIR

    Returns:
        df_summary : DataFrame with mean R and distance per channel per mode
    """
    if subject_ids is None:
        subject_ids = SUBJECT_IDS

    channels  = CHANNELS
    distances = [MEAN_DIST[ch] for ch in channels]

    # Collect all R values per channel per mode across subjects and tasks
    r_records = []
    for sid in subject_ids:
        for mode in ['original', 'derivative']:
            df = load_avg_csv(sid, mode)
            if df is None:
                continue
            df_model = df[df['Model'] == model_name]
            for _, row in df_model.iterrows():
                ch = row['Target_CH']
                if ch not in channels:
                    continue
                r_records.append({
                    'Subject': sid,
                    'Mode'   : mode,
                    'Channel': ch,
                    'Task'   : row['Task'],
                    'R'      : row['R'],
                })

    if not r_records:
        print("[ERROR] No data found. Run training first.")
        return None

    df_all = pd.DataFrame(r_records)

    # Mean R per channel per mode (across all subjects and tasks)
    mean_r = (df_all
              .groupby(['Mode', 'Channel'])['R']
              .mean()
              .round(4)
              .unstack('Channel')
              .reindex(columns=channels))

    mean_r_orig  = mean_r.loc['original'].values   if 'original'   in mean_r.index else None
    mean_r_deriv = mean_r.loc['derivative'].values  if 'derivative' in mean_r.index else None

    # ==========================================================================
    # Print results
    # ==========================================================================
    print(f"\n{'='*60}")
    print(f"  Mean R ({model_name}) per Channel")
    print(f"  Averaged across {len(subject_ids)} subject(s) and all tasks")
    print(f"{'='*60}")
    print(f"\n  {'Channel':<8} {'Distance':>10} {'R_Actual':>12} {'R_Derivative':>14}")
    print(f"  {'-'*48}")
    for i, ch in enumerate(channels):
        r_o = f"{mean_r_orig[i]:.4f}"  if mean_r_orig  is not None else "  N/A"
        r_d = f"{mean_r_deriv[i]:.4f}" if mean_r_deriv is not None else "  N/A"
        print(f"  {ch:<8} {distances[i]:>10.4f} {r_o:>12} {r_d:>14}")

    # ==========================================================================
    # Pearson correlation: distance vs mean R
    # ==========================================================================
    print(f"\n{'='*60}")
    print(f"  Pearson Correlation: Distance vs Mean R")
    print(f"{'='*60}")

    corr_results = {}
    for mode_label, r_vals in [('Actual Signal',              mean_r_orig),
                                ('Actual + Derivative Signal', mean_r_deriv)]:
        if r_vals is None:
            continue
        r_corr, p_val = pearsonr(distances, r_vals)
        coeffs        = np.polyfit(distances, r_vals, 1)
        sig           = 'significant' if p_val < 0.05 else 'not significant'
        print(f"\n  Mode: {mode_label}")
        print(f"    Pearson r = {r_corr:.4f}")
        print(f"    p-value   = {p_val:.4f}  ({sig} at α=0.05)")
        print(f"    Regression: R = {coeffs[0]:.4f} × dist + {coeffs[1]:.4f}")
        corr_results[mode_label] = {
            'pearson_r': round(r_corr, 4),
            'p_value'  : round(p_val,  4),
            'slope'    : round(coeffs[0], 4),
            'intercept': round(coeffs[1], 4),
        }

    # ==========================================================================
    # Build summary DataFrame
    # ==========================================================================
    df_summary = pd.DataFrame({
        'Channel'           : channels,
        'Mean_Distance'     : distances,
        'Mean_R_Actual'     : mean_r_orig  if mean_r_orig  is not None else [np.nan]*len(channels),
        'Mean_R_Derivative' : mean_r_deriv if mean_r_deriv is not None else [np.nan]*len(channels),
    })
    for mode_label, res in corr_results.items():
        suffix = 'Actual' if 'Derivative' not in mode_label else 'Derivative'
        df_summary[f'Pearson_r_{suffix}']  = res['pearson_r']
        df_summary[f'p_value_{suffix}']    = res['p_value']
        df_summary[f'slope_{suffix}']      = res['slope']
        df_summary[f'intercept_{suffix}']  = res['intercept']

    csv_out = os.path.join(OUTPUT_DIR,
                           f'distance_r_correlation_{model_name}.csv')
    df_summary.to_csv(csv_out, index=False)
    print(f"\n  Summary CSV saved: {csv_out}")

    # ==========================================================================
    # Plot — dual Y-axis bar + line (matching the reference figure style)
    # ==========================================================================
    x         = np.arange(len(channels))
    bar_width = 0.35
    fig, ax1  = plt.subplots(figsize=(9, 6))

    # Bars — Mean R
    if mean_r_orig is not None:
        ax1.bar(x - bar_width/2, mean_r_orig, bar_width,
                label='Mean R (ANN) Actual Signal',
                color='#4472C4', alpha=0.9, zorder=3)
    if mean_r_deriv is not None:
        ax1.bar(x + bar_width/2, mean_r_deriv, bar_width,
                label='Mean R (ANN) Actual & Derivative Signal',
                color='#ED7D31', alpha=0.9, zorder=3)

    ax1.set_xlabel('Target Channel', fontsize=12)
    ax1.set_ylabel('Mean R', fontsize=12)
    ax1.set_ylim(0, 1.0)
    ax1.set_xticks(x)
    ax1.set_xticklabels(channels, fontsize=11)
    ax1.yaxis.set_major_locator(plt.MultipleLocator(0.2))
    ax1.grid(True, axis='y', linestyle='--', alpha=0.4, zorder=0)
    ax1.spines['top'].set_visible(False)

    # Line — Mean Distance (secondary Y-axis)
    ax2 = ax1.twinx()
    ax2.plot(x, distances, color='#70AD47', linewidth=2.0,
             marker='o', markersize=6, label='Mean Distance',
             zorder=5)
    ax2.set_ylabel('Mean Distance (polar units)', fontsize=12)
    ax2.set_ylim(0, 1.0)
    ax2.yaxis.set_major_locator(plt.MultipleLocator(0.2))
    ax2.spines['top'].set_visible(False)

    # Title
    n_subs = len(subject_ids)
    ax1.set_title(
        f'Mean Pearson R ({model_name}) and Mean Spatial Distance\n'
        f'per Target Channel | {n_subs} Subject(s) | All Tasks',
        fontsize=12, fontweight='bold', pad=12
    )

    # Combined legend
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(handles1 + handles2, labels1 + labels2,
               loc='upper center', bbox_to_anchor=(0.5, -0.12),
               ncol=1, fontsize=10, frameon=True, edgecolor='#CCCCCC')

    plt.tight_layout()

    if save_fig:
        fig_path = os.path.join(OUTPUT_DIR,
                                f'distance_r_bar_{model_name}.png')
        fig.savefig(fig_path, dpi=300, bbox_inches='tight',
                    facecolor='white')
        print(f"  Figure saved: {fig_path}")

    plt.show()

    return df_summary


# =============================================================================
# MAIN
# =============================================================================

if __name__ == '__main__':
    df = compute_distance_r_correlation(
        subject_ids=SUBJECT_IDS,
        model_name=MODEL_NAME,
        save_fig=True
    )

In [ ]:
# =============================================================================
# statistical_analysis.py
#
# Statistical tests for EEG channel estimation results.
# Designed to satisfy IEEE Access reviewer requirements.
#
# Tests performed:
#   1. Model comparison (ANN vs RF, BDTR, SVR)
#      - Paired t-test (if normality holds)
#      - Wilcoxon signed-rank test (non-parametric, always reported)
#      - Effect size: Cohen's d
#
#   2. Mode comparison (original vs derivative, per model)
#      - Paired t-test + Wilcoxon signed-rank test
#      - Effect size: Cohen's d
#
#   3. Distance vs Mean R correlation significance
#      - Pearson r with n=5 channels, across all subjects
#      - Note on sample size limitation
#
# Input : averaged CSV per subject per mode
# Output: console summary + CSV report
# =============================================================================

import os
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations

# =============================================================================
# CONFIG
# =============================================================================
OUTPUT_DIR  = '/content/drive/MyDrive/Thesis/channel_estimation'
SUBJECT_IDS = [1,2,3,4,5,6,7,8,9,10]   # extend to all subjects, e.g. [1, 2, ..., 10]
ALPHA       = 0.05
MODELS      = ['ANN_MLP', 'Random_Forest', 'BDTR', 'SVR']
CHANNELS    = ['FZ', 'FC2', 'CZ', 'CP2', 'P4']
TASKS       = ['Arithmetic', 'Stroop', 'Mirror_image']

MEAN_DIST = {
    'FZ' : 0.3617, 'FC2': 0.4501,
    'CZ' : 0.5360, 'CP2': 0.6564, 'P4': 0.7946,
}


# =============================================================================
# HELPERS
# =============================================================================

def load_avg_csv(subject_id, input_mode):
    label = {'original': 'Actual', 'derivative': 'Derivative'}[input_mode]
    path  = os.path.join(OUTPUT_DIR,
                         f"results_{label}_sub{subject_id}_averaged.csv")
    if not os.path.exists(path):
        return None
    return pd.read_csv(path)


def cohens_d(a, b):
    """Cohen's d for paired samples."""
    diff   = np.array(a) - np.array(b)
    pooled = np.std(diff, ddof=1)
    return round(float(np.mean(diff) / pooled), 4) if pooled != 0 else 0.0


def effect_size_label(d):
    d = abs(d)
    if d < 0.2:   return 'negligible'
    if d < 0.5:   return 'small'
    if d < 0.8:   return 'medium'
    return 'large'


def run_paired_tests(a, b, label_a, label_b):
    """
    Run paired t-test + Wilcoxon signed-rank test on two paired samples.
    Returns a result dict and prints a formatted summary.
    """
    a, b = np.array(a), np.array(b)
    n    = len(a)
    diff = a - b

    # Shapiro-Wilk normality test on differences
    if n >= 3:
        _, p_norm = stats.shapiro(diff)
        normal    = p_norm > ALPHA
    else:
        p_norm, normal = np.nan, False

    # Paired t-test
    t_stat, p_ttest = stats.ttest_rel(a, b)

    # Wilcoxon signed-rank (requires at least some non-zero differences)
    try:
        w_stat, p_wilcox = stats.wilcoxon(diff, zero_method='wilcox',
                                           alternative='two-sided')
    except ValueError:
        w_stat, p_wilcox = np.nan, np.nan

    d    = cohens_d(a, b)
    d_lbl = effect_size_label(d)
    mean_diff = round(float(np.mean(diff)), 4)

    result = {
        'comparison'      : f"{label_a} vs {label_b}",
        'n'               : n,
        'mean_diff'       : mean_diff,
        'mean_a'          : round(float(np.mean(a)), 4),
        'mean_b'          : round(float(np.mean(b)), 4),
        'shapiro_p'       : round(float(p_norm), 4) if not np.isnan(p_norm) else np.nan,
        'normality'       : 'yes' if normal else 'no',
        't_stat'          : round(float(t_stat), 4),
        'p_ttest'         : round(float(p_ttest), 4),
        'sig_ttest'       : p_ttest < ALPHA,
        'w_stat'          : round(float(w_stat), 4) if not np.isnan(w_stat) else np.nan,
        'p_wilcox'        : round(float(p_wilcox), 4) if not np.isnan(p_wilcox) else np.nan,
        'sig_wilcox'      : (p_wilcox < ALPHA) if not np.isnan(p_wilcox) else False,
        'cohens_d'        : d,
        'effect_size'     : d_lbl,
    }

    sig_t = '*' if result['sig_ttest']  else 'ns'
    sig_w = '*' if result['sig_wilcox'] else 'ns'
    print(f"    {label_a:20s} vs {label_b:20s} | n={n}")
    print(f"      Mean: {np.mean(a):.4f} vs {np.mean(b):.4f}  "
          f"(diff={mean_diff:+.4f})")
    print(f"      t-test : t={t_stat:.3f}, p={p_ttest:.4f} [{sig_t}]")
    print(f"      Wilcoxon: W={w_stat:.1f}, p={p_wilcox:.4f} [{sig_w}]"
          if not np.isnan(w_stat) else "      Wilcoxon: not computable")
    print(f"      Cohen's d = {d:.4f} ({d_lbl})")
    print(f"      Normality (Shapiro-Wilk): p={p_norm:.4f} "
          f"({'normal' if normal else 'non-normal'})")

    return result


# =============================================================================
# TEST 1: Model comparison (per mode, pooled across subjects, channels, tasks)
# =============================================================================

def test_model_comparison(df_all, mode):
    """
    For a given mode, compare each model pair using paired tests.
    Pairing unit: one R value per (subject, channel, task) combination.
    """
    label = {'original': 'Actual Signal',
             'derivative': 'Actual + Derivative Signal'}[mode]
    df    = df_all[df_all['Mode'] == mode]

    print(f"\n{'='*60}")
    print(f"  TEST 1: Model Comparison | {label}")
    print(f"{'='*60}")

    results = []
    # Pivot to get one row per (subject, channel, task), columns = models
    pivot = df.pivot_table(
        index=['Subject', 'Target_CH', 'Task'],
        columns='Model', values='R'
    ).dropna()

    if pivot.empty:
        print("  [SKIP] Not enough data for model comparison.")
        return results

    print(f"  Pairs (n = {len(pivot)} observations per model):")
    # ANN_MLP vs all others
    for other in ['Random_Forest', 'BDTR', 'SVR']:
        if 'ANN_MLP' not in pivot.columns or other not in pivot.columns:
            continue
        print()
        r = run_paired_tests(pivot['ANN_MLP'].values,
                             pivot[other].values,
                             'ANN_MLP', other)
        r['mode'] = mode
        r['test'] = 'model_comparison'
        results.append(r)

    # All model pairs
    print(f"\n  All pairwise comparisons:")
    for m1, m2 in combinations(MODELS, 2):
        if m1 not in pivot.columns or m2 not in pivot.columns:
            continue
        if m1 == 'ANN_MLP':  # already done above
            continue
        print()
        r = run_paired_tests(pivot[m1].values, pivot[m2].values, m1, m2)
        r['mode'] = mode
        r['test'] = 'model_comparison'
        results.append(r)

    return results


# =============================================================================
# TEST 2: Mode comparison (original vs derivative, per model)
# =============================================================================

def test_mode_comparison(df_orig, df_deriv):
    """
    For each model, compare R values between original and derivative modes.
    Pairing unit: one R value per (subject, channel, task).
    """
    print(f"\n{'='*60}")
    print(f"  TEST 2: Mode Comparison (Original vs Derivative)")
    print(f"{'='*60}")

    results = []
    # Merge on common keys
    df_o = df_orig.copy();  df_o['Mode'] = 'original'
    df_d = df_deriv.copy(); df_d['Mode'] = 'derivative'
    df_merged = pd.merge(
        df_o[['Subject', 'Target_CH', 'Task', 'Model', 'R']],
        df_d[['Subject', 'Target_CH', 'Task', 'Model', 'R']],
        on=['Subject', 'Target_CH', 'Task', 'Model'],
        suffixes=('_orig', '_deriv')
    ).dropna()

    if df_merged.empty:
        print("  [SKIP] Not enough data for mode comparison.")
        return results

    for model in MODELS:
        sub = df_merged[df_merged['Model'] == model]
        if sub.empty:
            continue
        print(f"\n  Model: {model} (n = {len(sub)} pairs)")
        r = run_paired_tests(sub['R_orig'].values,
                             sub['R_deriv'].values,
                             'Actual Signal',
                             'Actual + Derivative')
        r['model'] = model
        r['test']  = 'mode_comparison'
        results.append(r)

    return results


# =============================================================================
# TEST 3: Distance vs Mean R correlation
# =============================================================================

def test_distance_correlation(df_all):
    """
    Correlate mean spatial distance with mean R (ANN_MLP) per channel.
    Reports Pearson r, p-value, and notes sample size limitation (n=5).
    """
    print(f"\n{'='*60}")
    print(f"  TEST 3: Distance vs Mean R Correlation (ANN_MLP)")
    print(f"{'='*60}")

    channels  = CHANNELS
    distances = [MEAN_DIST[ch] for ch in channels]
    results   = []

    for mode in ['original', 'derivative']:
        label  = {'original': 'Actual Signal',
                  'derivative': 'Actual + Derivative Signal'}[mode]
        df_m   = df_all[(df_all['Mode'] == mode) &
                        (df_all['Model'] == 'ANN_MLP')]

        mean_r = (df_m.groupby('Target_CH')['R']
                      .mean()
                      .reindex(channels))

        valid    = ~mean_r.isna()
        d_valid  = np.array(distances)[valid.values]
        r_valid  = mean_r.dropna().values

        if len(d_valid) < 3:
            continue

        r_corr, p_val = stats.pearsonr(d_valid, r_valid)
        coeffs        = np.polyfit(d_valid, r_valid, 1)
        sig           = p_val < ALPHA

        print(f"\n  Mode: {label} (n = {len(d_valid)} channels)")
        print(f"    Pearson r = {r_corr:.4f}")
        print(f"    p-value   = {p_val:.4f}  "
              f"({'*significant*' if sig else 'not significant'} at α={ALPHA})")
        print(f"    Regression: R = {coeffs[0]:.4f} × dist + {coeffs[1]:.4f}")
        if not sig:
            print(f"    Note: with n=5, threshold for significance is r ≥ 0.878.")
            print(f"          r={r_corr:.4f} indicates strong correlation "
                  f"despite p>{ALPHA}.")

        results.append({
            'test'     : 'distance_correlation',
            'mode'     : mode,
            'n'        : len(d_valid),
            'pearson_r': round(r_corr, 4),
            'p_value'  : round(p_val,  4),
            'sig'      : sig,
            'slope'    : round(coeffs[0], 4),
            'intercept': round(coeffs[1], 4),
        })

    return results


# =============================================================================
# FRAMING HELPER — generate paper-ready sentences
# =============================================================================

def generate_framing(model_results, mode_results, dist_results):
    """
    Print ready-to-use sentences for the paper based on test outcomes.
    """
    print(f"\n{'='*60}")
    print(f"  PAPER-READY FRAMING")
    print(f"{'='*60}")

    # Model comparison framing
    sig_pairs = [r for r in model_results if r.get('sig_wilcox')]
    if not sig_pairs:
        print("\n  [Model Comparison]")
        print("  No statistically significant difference was found between")
        print("  models at α=0.05 (Wilcoxon signed-rank test), consistent")
        print("  with the modest absolute differences reported in the results.")
    else:
        print("\n  [Model Comparison — Significant pairs found]")
        for r in sig_pairs:
            print(f"    {r['comparison']}: p={r['p_wilcox']:.4f}, "
                  f"d={r['cohens_d']} ({r['effect_size']})")

    # Mode comparison framing
    sig_modes = [r for r in mode_results if r.get('sig_wilcox')]
    if not sig_modes:
        print("\n  [Mode Comparison]")
        print("  No statistically significant difference was found between")
        print("  the original and derivative input modes at α=0.05,")
        print("  suggesting that adding first-order derivative features")
        print("  did not significantly alter estimation performance.")
    else:
        print("\n  [Mode Comparison — Significant differences found]")
        for r in sig_modes:
            direction = ('derivative > original'
                         if r['mean_diff'] < 0 else 'original > derivative')
            print(f"    {r['model']}: p={r['p_wilcox']:.4f}, "
                  f"d={r['cohens_d']} ({r['effect_size']}), {direction}")

    # Distance correlation framing
    print("\n  [Distance vs R Correlation]")
    for r in dist_results:
        label = ('Actual Signal' if r['mode'] == 'original'
                 else 'Actual + Derivative Signal')
        if r['sig']:
            print(f"  {label}: Significant negative correlation "
                  f"(r={r['pearson_r']:.4f}, p={r['p_value']:.4f}).")
        else:
            print(f"  {label}: Strong negative correlation "
                  f"(r={r['pearson_r']:.4f}, p={r['p_value']:.4f}), "
                  f"not significant at α=0.05 due to n=5 channels.")


# =============================================================================
# MAIN
# =============================================================================

def run_all_tests(subject_ids=None):
    """
    Run all statistical tests and save results to CSV.

    Parameters:
        subject_ids : list of subject IDs (default: SUBJECT_IDS)

    Returns:
        df_results : DataFrame with all test results
    """
    if subject_ids is None:
        subject_ids = SUBJECT_IDS

    print(f"\n{'#'*60}")
    print(f"  Statistical Analysis")
    print(f"  Subjects: {subject_ids} | α = {ALPHA}")
    print(f"{'#'*60}")

    # Load all data
    records = []
    for sid in subject_ids:
        for mode in ['original', 'derivative']:
            df = load_avg_csv(sid, mode)
            if df is None:
                print(f"  [WARN] CSV not found: sub{sid} {mode}")
                continue
            df['Subject'] = sid
            df['Mode']    = mode
            records.append(df)

    if not records:
        print("[ERROR] No data found. Run training first.")
        return None

    df_all   = pd.concat(records, ignore_index=True)
    df_orig  = df_all[df_all['Mode'] == 'original']
    df_deriv = df_all[df_all['Mode'] == 'derivative']

    all_results = []

    # Run tests
    model_results = (test_model_comparison(df_all, 'original') +
                     test_model_comparison(df_all, 'derivative'))
    mode_results  = test_mode_comparison(df_orig, df_deriv)
    dist_results  = test_distance_correlation(df_all)

    all_results = model_results + mode_results + dist_results

    # Paper-ready framing
    generate_framing(model_results, mode_results, dist_results)

    # Save to CSV
    df_results = pd.DataFrame(all_results)
    out_path   = os.path.join(OUTPUT_DIR, 'statistical_analysis_results.csv')
    df_results.to_csv(out_path, index=False)
    print(f"\n\n  Full results saved: {out_path}")

    return df_results


if __name__ == '__main__':
    df = run_all_tests(subject_ids=SUBJECT_IDS)